> # 가상환경 : NVIDIA GeForce RTX 3060에서 실행
- conda create -n EXAONE3_5 python=3.10 -y
- conda activate EXAONE3_5

- pip install torch==2.8.0+cu126 --index-url https://download.pytorch.org/whl/cu126
- pip uninstall -y torchvision torchaudio
- pip uninstall -y autoawq awq triton
- pip cache purge
- pip install "autoawq==0.2.9" --no-deps
- pip install "transformers==4.55.4" "accelerate==1.10.1" "sentencepiece==0.2.1" "tokenizers==0.21.4" "huggingface_hub==0.34.4" "safetensors==0.6.2" "numpy==2.0.2" "einops==0.8.1" "tqdm==4.67.1"
- pip install datasets

- cd "C:\Users\khu\Downloads\DW_PRSN"
- code .

> # Load Simulation Model : EXAONE-3.5-7.8B-Instruct-AWQ

In [ ]:
import torch
from transformers import pipeline, AutoTokenizer
MODEL_NAME = "LGAI-EXAONE/EXAONE-3.5-7.8B-Instruct-AWQ"
pipe = pipeline( task="text-generation", model=MODEL_NAME, tokenizer=MODEL_NAME, trust_remote_code=True,
                device_map="auto" if torch.cuda.is_available() else None, framework="pt", )
print("✅ pipeline ready")

# 메시지 형태 테스트
out = pipe([{"role":"user","content":"Who are you?"}], max_new_tokens=64, do_sample=False)
print(out[0]["generated_text"])

# **참치액**

> # 생성한 참치액 페르소나 불러오기

In [ ]:
import re, csv, os
from typing import List, Tuple

with open("chamchiaek.txt", "r", encoding="utf-8") as f:
    PERSONAS_RAW = f.read()

print("\n".join(PERSONAS_RAW.splitlines()[:5]))

# 속성별 가중치 계산 : 순위로

In [ ]:
# -*- coding: utf-8 -*-
import re
from typing import List, Dict, Tuple

# -----------------------------------------
# 0) 속성 정의
# -----------------------------------------
ATTRS: Dict[int, Tuple[str, str]] = {
    1: ("나이", "10대 / 20대 / 30대 / 40-50대 / 60대 이상"),
    2: ("성별", "남성 / 여성"),
    3: ("가구원수", "1인 / 2인(부부) / 2세대 이상(부모+자녀 등)"),
    4: ("자녀 유무", "있음 / 없음"),
    5: ("월평균 가구 소득", "저소득(200~350) / 중간(350~700) / 고소득(700초과) 만원"),
    6: ("구매 채널", "오프라인 / 온라인몰 / 온+오프라인"),
    7: ("최근 1주 외식·배달 횟수", "0회 / 1-2회 / 3-4회 / 5회 이상"),
    8: ("요리 여부·빈도", "전혀 안 함 / 가끔(주1-2) / 자주(주3+)"),
    9: ("선호 음식 카테고리", "국·찌개 / 볶음·양념 / 둘 다"),
    10: ("선호 참치액 브랜드", "동원 / 사조대림 / 한라 / 기타"),
}

# -----------------------------------------
# 옵션: 라운드별 로그 출력 여부 (기본 False)
# -----------------------------------------
VERBOSE = False

# -----------------------------------------
# 1) 프롬프트 생성기
#    - 모델 출력: 반드시 "속성n" (예: 속성10)
# -----------------------------------------
def build_single_pick_prompt(remaining_ids: List[int]) -> str:
    lines = []
    for i in remaining_ids:
        short, detail = ATTRS[i]
        lines.append(f"{i}. {short} — {detail}")
    attr_block = "\n".join(lines)

    prompt = f"""
너는 한국 소비자 조미료(참치액) 구매 행동을 분석하는 시장조사 전문가다.
아래 속성 후보들 중에서 '참치액 구매'에 가장 큰 영향을 미치는 속성 하나만 고르라.

[후보 속성]
{attr_block}

[출력 형식(다른 말 절대 금지)]
속성[번호]

예시:
속성10
""".strip()
    return prompt

# -----------------------------------------
# 2) 파서: "속성n" 한 줄만 허용
#    - (안전장치로 숫자만 응답해도 인식)
# -----------------------------------------
PAT_PROP = re.compile(r'^\s*속성\s*(\d{1,2})\s*$', re.IGNORECASE)
PAT_NUM  = re.compile(r'^\s*(\d{1,2})\s*$')

def parse_single_pick(text: str, allowed: List[int]) -> int:
    txt = text.strip()
    m = PAT_PROP.match(txt)
    if m:
        n = int(m.group(1))
        return n if n in allowed else -1
    m2 = PAT_NUM.match(txt)  # 예외적으로 숫자만 줄 경우
    if m2:
        n = int(m2.group(1))
        return n if n in allowed else -1
    return -1

# -----------------------------------------
# 3) 제거식 랭킹
# -----------------------------------------
def get_tuna_sauce_importance_elimination(pipe) -> List[int]:
    remaining = list(range(1, 11))
    ranking = []

    round_idx = 1
    while remaining:
        prompt = build_single_pick_prompt(remaining)
        out = pipe(
            [{"role": "user", "content": prompt}],
            max_new_tokens=8,          # "속성n"만 받으면 충분
            do_sample=False,           # 결정적 응답
            return_full_text=False
        )
        raw = out[0].get("generated_text", out[0].get("text", "")).strip()
        picked = parse_single_pick(raw, remaining)

        # 파싱 실패 시 보수적 폴백
        if picked == -1:
            picked = remaining[0]
            if VERBOSE:
                print(f"[라운드 {round_idx}] 파싱 실패 → 폴백 사용, 선택: 속성{picked}")

        if VERBOSE:
            print(f"[라운드 {round_idx}] 모델 출력: {raw}  → 선택: 속성{picked}")

        ranking.append(picked)
        remaining.remove(picked)
        round_idx += 1

    # 최종 요약만 출력
    print("\n[속성별 중요도 순위]  (1위 → 10위)")
    for pos, idx in enumerate(ranking, start=1):
        short, detail = ATTRS[idx]
        print(f"{pos}위: 속성{idx} — {short}")
    return ranking

# -----------------------------------------
# 4) 실행부 (이미 pipe 로드됨)
# -----------------------------------------
if __name__ == "__main__":
    ranking_elim = get_tuna_sauce_importance_elimination(pipe)
    # 필요하면 ranking_elim 리스트를 그대로 활용

# 월별 가중치를 위한 Simulation : 참치액 순 500g

In [ ]:
# -*- coding: utf-8 -*-
import re, csv, os
from typing import List, Tuple, Dict

# -----------------------------------------
# 0) 후보 달 (2024-04 ~ 2025-06)
# -----------------------------------------
ALLOWED_MONTHS: List[str] = [
    "2024-04","2024-05","2024-06","2024-07","2024-08","2024-09",
    "2024-10","2024-11","2024-12","2025-01","2025-02","2025-03",
    "2025-04","2025-05","2025-06"
]

# -----------------------------------------
# 1) 속성 중요도
# -----------------------------------------
IMPORTANCE_ORDER = [
    "최근 1주 외식·배달 횟수", "월평균 가구 소득", "선호 음식 카테고리",
    "요리 여부·빈도", "가구원수", "선호 참치액 브랜드",
    "구매 채널", "자녀 유무", "나이", "성별"
]

# -----------------------------------------
# 1-1) 대상 제품/설명
# -----------------------------------------
PRODUCT_NAME = "동원 참치액 순 500g"
PRODUCT_DESC = (
    "동원의 참치액은 훈연참치추출물 80% 이상과 참치엑기스를 사용한 액상 조미료이다. "
    "남태평양 등 청정해역에서 잡은 참치를 5시간 이상 끓여 만든 참치엑기스와 "
    "가쓰오엑기스, 표고버섯, 무, 감초, 마늘 등을 넣어 깊은 감칠맛을 완성한다. "
    "‘동원참치액 순’은 훈연 향을 줄이고 멸치 숙성액을 넣어 시원하고 깔끔한 맛을 구현한다."
)

# -----------------------------------------
# 2) 프롬프트 (대상 제품 한정)
# -----------------------------------------
def build_single_month_prompt(persona_text: str, months: List[str]) -> str:
    months_line = ", ".join(months)
    importance_line = " > ".join(IMPORTANCE_ORDER)
    return f"""
너는 한국 소비자 조미료(참치액) 구매 행동을 분석하는 시장조사 전문가다.

[제품명]
{PRODUCT_NAME}

[제품 설명]
{PRODUCT_DESC}

[페르소나]
{persona_text.strip()}

[질문]
아래 달 중에서 이 페르소나가 위의 제품({PRODUCT_NAME})을 가장 많이 살 것 같은 달 1개만 고르라.
판단 시 아래 속성 중요도(상→하)를 우선 고려하라: {importance_line}

[선택 가능한 달(반드시 이 중에서만 선택)]
{months_line}

[출력 형식(다른 말 절대 금지)]
YYYY-MM
""".strip()

# -----------------------------------------
# 3) 응답 파서
# -----------------------------------------
RE_ISO_LINE  = re.compile(r'^\s*(20\d{2})-(0[1-9]|1[0-2])\s*$', re.MULTILINE)
RE_KOR_LINE  = re.compile(r'^\s*(20\d{2})\s*년\s*(1[0-2]|[1-9])\s*월\s*$', re.MULTILINE)

def parse_one_month(text: str, allowed: List[str]) -> str:
    for y, m in RE_ISO_LINE.findall(text):
        cand = f"{y}-{m}"
        if cand in allowed:
            return cand
    for y, m in RE_KOR_LINE.findall(text):
        cand = f"{y}-{int(m):02d}"
        if cand in allowed:
            return cand
    m = re.search(r'(20\d{2})-(0[1-9]|1[0-2])', text)
    if m:
        cand = f"{m.group(1)}-{m.group(2)}"
        if cand in allowed:
            return cand
    return ""

# -----------------------------------------
# 4) 페르소나 TXT 파싱
# -----------------------------------------
BLOCK_RE = re.compile(
    r"\*\*Persona_(\d{1,3})\*\*\s*(.+?)(?=(?:\n\s*\*\*Persona_\d{1,3}\*\*|\Z))",
    re.DOTALL
)

def parse_personas(text: str) -> List[Tuple[str, str]]:
    personas: List[Tuple[str, str]] = []
    for pid, body in BLOCK_RE.findall(text):
        personas.append((f"Persona_{int(pid):03d}", body.strip()))
    return personas

# -----------------------------------------
# 5) 단일 페르소나 질의
# -----------------------------------------
def ask_month_for_persona(
    pipe,
    persona_text: str,
    temperature: float = 1.20,
    top_p: float = 0.97,
    top_k: int = 0,
    max_new_tokens: int = 12,
    verbose: bool = False
) -> Tuple[str, str]:
    prompt = build_single_month_prompt(persona_text, ALLOWED_MONTHS)
    out = pipe(
        [{"role": "user", "content": prompt}],
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=temperature,
        top_p=top_p,
        top_k=top_k,
        repetition_penalty=1.0,
        return_full_text=False
    )
    raw = out[0].get("generated_text", out[0].get("text", "")).strip()
    month = parse_one_month(raw, ALLOWED_MONTHS) or ""
    if verbose:
        print(f"[RAW] {raw} -> [PARSED] {month or '(fail)'}")
    return month, raw

# -----------------------------------------
# 6) 전체 실행 루프
# -----------------------------------------
def run_survey_over_personas(
    pipe,
    personas_text: str,
    save_dir: str = ".",
    show_progress: bool = True,
    print_details: bool = True
) -> Dict[str, int]:
    os.makedirs(save_dir, exist_ok=True)

    personas = parse_personas(personas_text)
    if not personas:
        raise ValueError("페르소나를 찾지 못했으니 입력 형식을 확인하라.")

    month_counts: Dict[str, int] = {m: 0 for m in ALLOWED_MONTHS}
    total = len(personas)

    log_path = os.path.join(save_dir, "persona_choices.csv")
    with open(log_path, "w", newline="", encoding="utf-8") as f:
        wr = csv.writer(f)
        wr.writerow(["persona_id", "chosen_month", "raw_output"])

        for i, (pid, body) in enumerate(personas, start=1):
            if show_progress:
                print(f"[{i}/{total}] {pid} 질문 중...", flush=True)

            try:
                chosen, raw = ask_month_for_persona(pipe, body, verbose=False)
            except Exception as e:
                chosen, raw = "", f"__ERROR__: {e!r}"

            if chosen:
                month_counts[chosen] += 1
            wr.writerow([pid, chosen, raw])

            if show_progress and print_details:
                short_raw = (raw[:80] + "…") if len(raw) > 80 else raw
                print(f"   ↳ 선택: {chosen or '(파싱 실패)'} | 원문: {short_raw}", flush=True)

    cnt_path = os.path.join(save_dir, "month_counts_soon_500g.csv")
    with open(cnt_path, "w", newline="", encoding="utf-8") as f:
        wr = csv.writer(f)
        wr.writerow(["month", "count"])
        for m in ALLOWED_MONTHS:
            wr.writerow([m, month_counts[m]])

    print("\n[월별 선택 횟수]")
    for m in ALLOWED_MONTHS:
        print(f"{m}: {month_counts[m]}")

    print(f"\n저장 완료:\n- {log_path}\n- {cnt_path}")
    return month_counts

# -----------------------------------------
# 7) 실행
# -----------------------------------------
if __name__ == "__main__":
    month_counts = run_survey_over_personas(
        pipe,
        PERSONAS_RAW,
        save_dir=".",
        show_progress=True,
        print_details=True
    )


> ### **최종 월별 가중치 계산**


### 1. 데이터 준비

- 월별로 “이 제품을 가장 많이 구매할 것 같다”고 선택된 횟수를 Simulation을 통해 생성
- 예를 들어 2024년 7월은 18명, 2024년 4월은 6명처럼 각 달마다 몇 번 선택됐는지가 정리돼 있다. 이 값이 바로 `count` 열

---

### 2. 평균 구하기

- 이 `count` 값들의 전체 평균 계산
- 즉, 모든 달의 선택 횟수를 더한 뒤, 달 수로 나눈 값이 `mean_count`
- 이 평균은 “각 달이 보통 몇 번 선택되는가”라는 기준선 역할을 함

---

### 3. factor 계산

- 각 달의 선택 횟수를 평균으로 나누어, 그 달이 평균보다 얼마나 많은지 또는 적은지를 비율로 표현

* 평균과 같으면 factor는 1
* 평균보다 많으면 1보다 크고
* 평균보다 적으면 1보다 작다
-   이 값이 바로 `factor`다.

---

### 4. adj\_factor 계산 (가중치)

- 이제 factor를 그대로 쓰지 않고, 평균에서 얼마나 벗어났는지를 활용하여 월별 가중치로 계산

* factor가 1일 때는 그대로 1이 된다.
* factor가 2라면, “평균의 두 배”라는 정보를 전부 반영하지 않고, 평균에서 1만큼 벗어난 걸 100분의 1만큼만 반영한다. 그래서 최종 값은 1.01 정도가 된다.
* factor가 0.5라면, “평균의 절반”이라는 차이를 크게 쓰지 않고, 평균에서 0.5만큼 벗어난 걸 100분의 1만큼만 반영해서 0.995 정도가 된다.

즉, `adj_factor`는 “평균(1)을 기준으로, factor와의 차이를 1% 정도만 반영한 값”이라고 보면 된다.

In [ ]:
import pandas as pd

# CSV 파일 불러오기
df = pd.read_csv("month_counts_soon_500g.csv", encoding="utf-8")

# 평균 count 계산
mean_count = df["count"].mean()

# factor 및 보정된 adj_factor 계산
df["factor"] = df["count"] / mean_count
df["adj_factor"] = 1 + (df["factor"] -1) / 100

print(df)

# 월별 가중치를 위한 Simulation : 참치액 순 900g

In [ ]:
# -*- coding: utf-8 -*-
import re, csv, os
from typing import List, Tuple, Dict

# -----------------------------------------
# 0) 후보 달 (2024-04 ~ 2025-06)
# -----------------------------------------
ALLOWED_MONTHS: List[str] = [
    "2024-04","2024-05","2024-06","2024-07","2024-08","2024-09",
    "2024-10","2024-11","2024-12","2025-01","2025-02","2025-03",
    "2025-04","2025-05","2025-06"
]

# -----------------------------------------
# 1) 속성 중요도
# -----------------------------------------
IMPORTANCE_ORDER = [
    "최근 1주 외식·배달 횟수", "월평균 가구 소득", "선호 음식 카테고리",
    "요리 여부·빈도", "가구원수", "선호 참치액 브랜드",
    "구매 채널", "자녀 유무", "나이", "성별"
]

# -----------------------------------------
# 1-1) 대상 제품/설명
# -----------------------------------------
PRODUCT_NAME = "동원 참치액 순 900g"
PRODUCT_DESC = (
    "동원의 참치액은 훈연참치추출물 80% 이상과 참치엑기스를 사용한 액상 조미료이다. "
    "남태평양 등 청정해역에서 잡은 참치를 5시간 이상 끓여 만든 참치엑기스와 "
    "가쓰오엑기스, 표고버섯, 무, 감초, 마늘 등을 넣어 깊은 감칠맛을 완성한다. "
    "‘동원참치액 순’은 훈연 향을 줄이고 멸치 숙성액을 넣어 시원하고 깔끔한 맛을 구현한다."
)

# -----------------------------------------
# 2) 프롬프트 (대상 제품 한정)
# -----------------------------------------
def build_single_month_prompt(persona_text: str, months: List[str]) -> str:
    months_line = ", ".join(months)
    importance_line = " > ".join(IMPORTANCE_ORDER)
    return f"""
너는 한국 소비자 조미료(참치액) 구매 행동을 분석하는 시장조사 전문가다.

[제품명]
{PRODUCT_NAME}

[제품 설명]
{PRODUCT_DESC}

[페르소나]
{persona_text.strip()}

[질문]
아래 달 중에서 이 페르소나가 위의 제품({PRODUCT_NAME})을 가장 많이 살 것 같은 달 1개만 고르라.
판단 시 아래 속성 중요도(상→하)를 우선 고려하라: {importance_line}

[선택 가능한 달(반드시 이 중에서만 선택)]
{months_line}

[출력 형식(다른 말 절대 금지)]
YYYY-MM
""".strip()

# -----------------------------------------
# 3) 응답 파서
# -----------------------------------------
RE_ISO_LINE  = re.compile(r'^\s*(20\d{2})-(0[1-9]|1[0-2])\s*$', re.MULTILINE)
RE_KOR_LINE  = re.compile(r'^\s*(20\d{2})\s*년\s*(1[0-2]|[1-9])\s*월\s*$', re.MULTILINE)

def parse_one_month(text: str, allowed: List[str]) -> str:
    for y, m in RE_ISO_LINE.findall(text):
        cand = f"{y}-{m}"
        if cand in allowed:
            return cand
    for y, m in RE_KOR_LINE.findall(text):
        cand = f"{y}-{int(m):02d}"
        if cand in allowed:
            return cand
    m = re.search(r'(20\d{2})-(0[1-9]|1[0-2])', text)
    if m:
        cand = f"{m.group(1)}-{m.group(2)}"
        if cand in allowed:
            return cand
    return ""

# -----------------------------------------
# 4) 페르소나 TXT 파싱
# -----------------------------------------
BLOCK_RE = re.compile(
    r"\*\*Persona_(\d{1,3})\*\*\s*(.+?)(?=(?:\n\s*\*\*Persona_\d{1,3}\*\*|\Z))",
    re.DOTALL
)

def parse_personas(text: str) -> List[Tuple[str, str]]:
    personas: List[Tuple[str, str]] = []
    for pid, body in BLOCK_RE.findall(text):
        personas.append((f"Persona_{int(pid):03d}", body.strip()))
    return personas

# -----------------------------------------
# 5) 단일 페르소나 질의
# -----------------------------------------
def ask_month_for_persona(
    pipe,
    persona_text: str,
    temperature: float = 1.20,
    top_p: float = 0.97,
    top_k: int = 0,
    max_new_tokens: int = 12,
    verbose: bool = False
) -> Tuple[str, str]:
    prompt = build_single_month_prompt(persona_text, ALLOWED_MONTHS)
    out = pipe(
        [{"role": "user", "content": prompt}],
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=temperature,
        top_p=top_p,
        top_k=top_k,
        repetition_penalty=1.0,
        return_full_text=False
    )
    raw = out[0].get("generated_text", out[0].get("text", "")).strip()
    month = parse_one_month(raw, ALLOWED_MONTHS) or ""
    if verbose:
        print(f"[RAW] {raw} -> [PARSED] {month or '(fail)'}")
    return month, raw

# -----------------------------------------
# 6) 전체 실행 루프
# -----------------------------------------
def run_survey_over_personas(
    pipe,
    personas_text: str,
    save_dir: str = ".",
    show_progress: bool = True,
    print_details: bool = True
) -> Dict[str, int]:
    os.makedirs(save_dir, exist_ok=True)

    personas = parse_personas(personas_text)
    if not personas:
        raise ValueError("페르소나를 찾지 못했으니 입력 형식을 확인하라.")

    month_counts: Dict[str, int] = {m: 0 for m in ALLOWED_MONTHS}
    total = len(personas)

    log_path = os.path.join(save_dir, "persona_choices.csv")
    with open(log_path, "w", newline="", encoding="utf-8") as f:
        wr = csv.writer(f)
        wr.writerow(["persona_id", "chosen_month", "raw_output"])

        for i, (pid, body) in enumerate(personas, start=1):
            if show_progress:
                print(f"[{i}/{total}] {pid} 질문 중...", flush=True)

            try:
                chosen, raw = ask_month_for_persona(pipe, body, verbose=False)
            except Exception as e:
                chosen, raw = "", f"__ERROR__: {e!r}"

            if chosen:
                month_counts[chosen] += 1
            wr.writerow([pid, chosen, raw])

            if show_progress and print_details:
                short_raw = (raw[:80] + "…") if len(raw) > 80 else raw
                print(f"   ↳ 선택: {chosen or '(파싱 실패)'} | 원문: {short_raw}", flush=True)

    cnt_path = os.path.join(save_dir, "month_counts_soon_900g.csv")
    with open(cnt_path, "w", newline="", encoding="utf-8") as f:
        wr = csv.writer(f)
        wr.writerow(["month", "count"])
        for m in ALLOWED_MONTHS:
            wr.writerow([m, month_counts[m]])

    print("\n[월별 선택 횟수]")
    for m in ALLOWED_MONTHS:
        print(f"{m}: {month_counts[m]}")

    print(f"\n저장 완료:\n- {log_path}\n- {cnt_path}")
    return month_counts

# -----------------------------------------
# 7) 실행
# -----------------------------------------
if __name__ == "__main__":
    month_counts = run_survey_over_personas(
        pipe,
        PERSONAS_RAW,
        save_dir=".",
        show_progress=True,
        print_details=True
    )


In [ ]:
import pandas as pd

# CSV 파일 불러오기
df = pd.read_csv("month_counts_soon_900g.csv", encoding="utf-8")

# 평균 count 계산
mean_count = df["count"].mean()

# factor 및 보정된 adj_factor 계산
df["factor"] = df["count"] / mean_count
df["adj_factor"] = 1 + (df["factor"] -1) / 100

print(df)

# 월별 가중치를 위한 Simulation : 참치액 진 500g

In [ ]:
# -*- coding: utf-8 -*-
import re, csv, os
from typing import List, Tuple, Dict

# -----------------------------------------
# 0) 후보 달 (2024-04 ~ 2025-06)
# -----------------------------------------
ALLOWED_MONTHS: List[str] = [
    "2024-04","2024-05","2024-06","2024-07","2024-08","2024-09",
    "2024-10","2024-11","2024-12","2025-01","2025-02","2025-03",
    "2025-04","2025-05","2025-06"
]

# -----------------------------------------
# 1) 속성 중요도
# -----------------------------------------
IMPORTANCE_ORDER = [
    "최근 1주 외식·배달 횟수", "월평균 가구 소득", "선호 음식 카테고리",
    "요리 여부·빈도", "가구원수", "선호 참치액 브랜드",
    "구매 채널", "자녀 유무", "나이", "성별"
]

# -----------------------------------------
# 1-1) 대상 제품/설명
# -----------------------------------------
PRODUCT_NAME = "동원 참치액 진 500g"
PRODUCT_DESC = (
    "동원의 참치액은 훈연참치추출물 80% 이상과 참치엑기스를 사용한 액상 조미료이다. "
    "남태평양 등 청정해역에서 잡은 참치를 5시간 이상 끓여 만든 참치엑기스와 "
    "가쓰오엑기스, 표고버섯, 무, 감초, 마늘 등을 넣어 깊은 감칠맛을 완성한다. "
    "‘동원참치액 진’은 진한 가쓰오 풍미가 특징이라 국물·조림·볶음 등 다양한 요리에 잘 어울린다."
)

# -----------------------------------------
# 2) 프롬프트 (대상 제품 한정)
# -----------------------------------------
def build_single_month_prompt(persona_text: str, months: List[str]) -> str:
    months_line = ", ".join(months)
    importance_line = " > ".join(IMPORTANCE_ORDER)
    return f"""
너는 한국 소비자 조미료(참치액) 구매 행동을 분석하는 시장조사 전문가다.

[제품명]
{PRODUCT_NAME}

[제품 설명]
{PRODUCT_DESC}

[페르소나]
{persona_text.strip()}

[질문]
아래 달 중에서 이 페르소나가 위의 제품({PRODUCT_NAME})을 가장 많이 살 것 같은 달 1개만 고르라.
판단 시 아래 속성 중요도(상→하)를 우선 고려하라: {importance_line}

[선택 가능한 달(반드시 이 중에서만 선택)]
{months_line}

[출력 형식(다른 말 절대 금지)]
YYYY-MM
""".strip()

# -----------------------------------------
# 3) 응답 파서
# -----------------------------------------
RE_ISO_LINE  = re.compile(r'^\s*(20\d{2})-(0[1-9]|1[0-2])\s*$', re.MULTILINE)
RE_KOR_LINE  = re.compile(r'^\s*(20\d{2})\s*년\s*(1[0-2]|[1-9])\s*월\s*$', re.MULTILINE)

def parse_one_month(text: str, allowed: List[str]) -> str:
    for y, m in RE_ISO_LINE.findall(text):
        cand = f"{y}-{m}"
        if cand in allowed:
            return cand
    for y, m in RE_KOR_LINE.findall(text):
        cand = f"{y}-{int(m):02d}"
        if cand in allowed:
            return cand
    m = re.search(r'(20\d{2})-(0[1-9]|1[0-2])', text)
    if m:
        cand = f"{m.group(1)}-{m.group(2)}"
        if cand in allowed:
            return cand
    return ""

# -----------------------------------------
# 4) 페르소나 TXT 파싱
# -----------------------------------------
BLOCK_RE = re.compile(
    r"\*\*Persona_(\d{1,3})\*\*\s*(.+?)(?=(?:\n\s*\*\*Persona_\d{1,3}\*\*|\Z))",
    re.DOTALL
)

def parse_personas(text: str) -> List[Tuple[str, str]]:
    personas: List[Tuple[str, str]] = []
    for pid, body in BLOCK_RE.findall(text):
        personas.append((f"Persona_{int(pid):03d}", body.strip()))
    return personas

# -----------------------------------------
# 5) 단일 페르소나 질의
# -----------------------------------------
def ask_month_for_persona(
    pipe,
    persona_text: str,
    temperature: float = 1.20,
    top_p: float = 0.97,
    top_k: int = 0,
    max_new_tokens: int = 12,
    verbose: bool = False
) -> Tuple[str, str]:
    prompt = build_single_month_prompt(persona_text, ALLOWED_MONTHS)
    out = pipe(
        [{"role": "user", "content": prompt}],
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=temperature,
        top_p=top_p,
        top_k=top_k,
        repetition_penalty=1.0,
        return_full_text=False
    )
    raw = out[0].get("generated_text", out[0].get("text", "")).strip()
    month = parse_one_month(raw, ALLOWED_MONTHS) or ""
    if verbose:
        print(f"[RAW] {raw} -> [PARSED] {month or '(fail)'}")
    return month, raw

# -----------------------------------------
# 6) 전체 실행 루프
# -----------------------------------------
def run_survey_over_personas(
    pipe,
    personas_text: str,
    save_dir: str = ".",
    show_progress: bool = True,
    print_details: bool = True
) -> Dict[str, int]:
    os.makedirs(save_dir, exist_ok=True)

    personas = parse_personas(personas_text)
    if not personas:
        raise ValueError("페르소나를 찾지 못했으니 입력 형식을 확인하라.")

    month_counts: Dict[str, int] = {m: 0 for m in ALLOWED_MONTHS}
    total = len(personas)

    log_path = os.path.join(save_dir, "persona_choices.csv")
    with open(log_path, "w", newline="", encoding="utf-8") as f:
        wr = csv.writer(f)
        wr.writerow(["persona_id", "chosen_month", "raw_output"])

        for i, (pid, body) in enumerate(personas, start=1):
            if show_progress:
                print(f"[{i}/{total}] {pid} 질문 중...", flush=True)

            try:
                chosen, raw = ask_month_for_persona(pipe, body, verbose=False)
            except Exception as e:
                chosen, raw = "", f"__ERROR__: {e!r}"

            if chosen:
                month_counts[chosen] += 1
            wr.writerow([pid, chosen, raw])

            if show_progress and print_details:
                short_raw = (raw[:80] + "…") if len(raw) > 80 else raw
                print(f"   ↳ 선택: {chosen or '(파싱 실패)'} | 원문: {short_raw}", flush=True)

    cnt_path = os.path.join(save_dir, "month_counts_jin_500g.csv")
    with open(cnt_path, "w", newline="", encoding="utf-8") as f:
        wr = csv.writer(f)
        wr.writerow(["month", "count"])
        for m in ALLOWED_MONTHS:
            wr.writerow([m, month_counts[m]])

    print("\n[월별 선택 횟수]")
    for m in ALLOWED_MONTHS:
        print(f"{m}: {month_counts[m]}")

    print(f"\n저장 완료:\n- {log_path}\n- {cnt_path}")
    return month_counts

# -----------------------------------------
# 7) 실행
# -----------------------------------------
if __name__ == "__main__":
    month_counts = run_survey_over_personas(
        pipe,
        PERSONAS_RAW,
        save_dir=".",
        show_progress=True,
        print_details=True
    )


In [ ]:
import pandas as pd

# CSV 파일 불러오기
df = pd.read_csv("month_counts_jin_500g.csv", encoding="utf-8")

# 평균 count 계산
mean_count = df["count"].mean()

# factor 및 보정된 adj_factor 계산
df["factor"] = df["count"] / mean_count
df["adj_factor"] = 1 + (df["factor"] -1) / 100

print(df)

# 월별 가중치를 위한 Simulation : 참치액 진 900g

In [ ]:
# -*- coding: utf-8 -*-
import re, csv, os
from typing import List, Tuple, Dict

# -----------------------------------------
# 0) 후보 달 (2024-04 ~ 2025-06)
# -----------------------------------------
ALLOWED_MONTHS: List[str] = [
    "2024-04","2024-05","2024-06","2024-07","2024-08","2024-09",
    "2024-10","2024-11","2024-12","2025-01","2025-02","2025-03",
    "2025-04","2025-05","2025-06"
]

# -----------------------------------------
# 1) 속성 중요도
# -----------------------------------------
IMPORTANCE_ORDER = [
    "최근 1주 외식·배달 횟수", "월평균 가구 소득", "선호 음식 카테고리",
    "요리 여부·빈도", "가구원수", "선호 참치액 브랜드",
    "구매 채널", "자녀 유무", "나이", "성별"
]

# -----------------------------------------
# 1-1) 대상 제품/설명
# -----------------------------------------
PRODUCT_NAME = "동원 참치액 진 900g"
PRODUCT_DESC = (
    "동원의 참치액은 훈연참치추출물 80% 이상과 참치엑기스를 사용한 액상 조미료이다. "
    "남태평양 등 청정해역에서 잡은 참치를 5시간 이상 끓여 만든 참치엑기스와 "
    "가쓰오엑기스, 표고버섯, 무, 감초, 마늘 등을 넣어 깊은 감칠맛을 완성한다. "
    "‘동원참치액 진’은 진한 가쓰오 풍미가 특징이라 국물·조림·볶음 등 다양한 요리에 잘 어울린다."
)

# -----------------------------------------
# 2) 프롬프트 (대상 제품 한정)
# -----------------------------------------
def build_single_month_prompt(persona_text: str, months: List[str]) -> str:
    months_line = ", ".join(months)
    importance_line = " > ".join(IMPORTANCE_ORDER)
    return f"""
너는 한국 소비자 조미료(참치액) 구매 행동을 분석하는 시장조사 전문가다.

[제품명]
{PRODUCT_NAME}

[제품 설명]
{PRODUCT_DESC}

[페르소나]
{persona_text.strip()}

[질문]
아래 달 중에서 이 페르소나가 위의 제품({PRODUCT_NAME})을 가장 많이 살 것 같은 달 1개만 고르라.
판단 시 아래 속성 중요도(상→하)를 우선 고려하라: {importance_line}

[선택 가능한 달(반드시 이 중에서만 선택)]
{months_line}

[출력 형식(다른 말 절대 금지)]
YYYY-MM
""".strip()

# -----------------------------------------
# 3) 응답 파서
# -----------------------------------------
RE_ISO_LINE  = re.compile(r'^\s*(20\d{2})-(0[1-9]|1[0-2])\s*$', re.MULTILINE)
RE_KOR_LINE  = re.compile(r'^\s*(20\d{2})\s*년\s*(1[0-2]|[1-9])\s*월\s*$', re.MULTILINE)

def parse_one_month(text: str, allowed: List[str]) -> str:
    for y, m in RE_ISO_LINE.findall(text):
        cand = f"{y}-{m}"
        if cand in allowed:
            return cand
    for y, m in RE_KOR_LINE.findall(text):
        cand = f"{y}-{int(m):02d}"
        if cand in allowed:
            return cand
    m = re.search(r'(20\d{2})-(0[1-9]|1[0-2])', text)
    if m:
        cand = f"{m.group(1)}-{m.group(2)}"
        if cand in allowed:
            return cand
    return ""

# -----------------------------------------
# 4) 페르소나 TXT 파싱
# -----------------------------------------
BLOCK_RE = re.compile(
    r"\*\*Persona_(\d{1,3})\*\*\s*(.+?)(?=(?:\n\s*\*\*Persona_\d{1,3}\*\*|\Z))",
    re.DOTALL
)

def parse_personas(text: str) -> List[Tuple[str, str]]:
    personas: List[Tuple[str, str]] = []
    for pid, body in BLOCK_RE.findall(text):
        personas.append((f"Persona_{int(pid):03d}", body.strip()))
    return personas

# -----------------------------------------
# 5) 단일 페르소나 질의
# -----------------------------------------
def ask_month_for_persona(
    pipe,
    persona_text: str,
    temperature: float = 1.20,
    top_p: float = 0.97,
    top_k: int = 0,
    max_new_tokens: int = 12,
    verbose: bool = False
) -> Tuple[str, str]:
    prompt = build_single_month_prompt(persona_text, ALLOWED_MONTHS)
    out = pipe(
        [{"role": "user", "content": prompt}],
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=temperature,
        top_p=top_p,
        top_k=top_k,
        repetition_penalty=1.0,
        return_full_text=False
    )
    raw = out[0].get("generated_text", out[0].get("text", "")).strip()
    month = parse_one_month(raw, ALLOWED_MONTHS) or ""
    if verbose:
        print(f"[RAW] {raw} -> [PARSED] {month or '(fail)'}")
    return month, raw

# -----------------------------------------
# 6) 전체 실행 루프
# -----------------------------------------
def run_survey_over_personas(
    pipe,
    personas_text: str,
    save_dir: str = ".",
    show_progress: bool = True,
    print_details: bool = True
) -> Dict[str, int]:
    os.makedirs(save_dir, exist_ok=True)

    personas = parse_personas(personas_text)
    if not personas:
        raise ValueError("페르소나를 찾지 못했으니 입력 형식을 확인하라.")

    month_counts: Dict[str, int] = {m: 0 for m in ALLOWED_MONTHS}
    total = len(personas)

    log_path = os.path.join(save_dir, "persona_choices.csv")
    with open(log_path, "w", newline="", encoding="utf-8") as f:
        wr = csv.writer(f)
        wr.writerow(["persona_id", "chosen_month", "raw_output"])

        for i, (pid, body) in enumerate(personas, start=1):
            if show_progress:
                print(f"[{i}/{total}] {pid} 질문 중...", flush=True)

            try:
                chosen, raw = ask_month_for_persona(pipe, body, verbose=False)
            except Exception as e:
                chosen, raw = "", f"__ERROR__: {e!r}"

            if chosen:
                month_counts[chosen] += 1
            wr.writerow([pid, chosen, raw])

            if show_progress and print_details:
                short_raw = (raw[:80] + "…") if len(raw) > 80 else raw
                print(f"   ↳ 선택: {chosen or '(파싱 실패)'} | 원문: {short_raw}", flush=True)

    cnt_path = os.path.join(save_dir, "month_counts_jin_900g.csv")
    with open(cnt_path, "w", newline="", encoding="utf-8") as f:
        wr = csv.writer(f)
        wr.writerow(["month", "count"])
        for m in ALLOWED_MONTHS:
            wr.writerow([m, month_counts[m]])

    print("\n[월별 선택 횟수]")
    for m in ALLOWED_MONTHS:
        print(f"{m}: {month_counts[m]}")

    print(f"\n저장 완료:\n- {log_path}\n- {cnt_path}")
    return month_counts

# -----------------------------------------
# 7) 실행
# -----------------------------------------
if __name__ == "__main__":
    month_counts = run_survey_over_personas(
        pipe,
        PERSONAS_RAW,
        save_dir=".",
        show_progress=True,
        print_details=True
    )


In [ ]:
import pandas as pd

# CSV 파일 불러오기
df = pd.read_csv("month_counts_jin_900g.csv", encoding="utf-8")

# 평균 count 계산
mean_count = df["count"].mean()

# factor 및 보정된 adj_factor 계산
df["factor"] = df["count"] / mean_count
df["adj_factor"] = 1 + (df["factor"] -1) / 100

print(df)

# 월별 가중치를 위한 Simulation : 프리미엄 동원참치액 500g

In [ ]:
# -*- coding: utf-8 -*-
import re, csv, os
from typing import List, Tuple, Dict

# -----------------------------------------
# 0) 후보 달 (2024-04 ~ 2025-06)
# -----------------------------------------
ALLOWED_MONTHS: List[str] = [
    "2024-04","2024-05","2024-06","2024-07","2024-08","2024-09",
    "2024-10","2024-11","2024-12","2025-01","2025-02","2025-03",
    "2025-04","2025-05","2025-06"
]

# -----------------------------------------
# 1) 속성 중요도
# -----------------------------------------
IMPORTANCE_ORDER = [
    "최근 1주 외식·배달 횟수", "월평균 가구 소득", "선호 음식 카테고리",
    "요리 여부·빈도", "가구원수", "선호 참치액 브랜드",
    "구매 채널", "자녀 유무", "나이", "성별"
]

# -----------------------------------------
# 1-1) 대상 제품/설명
# -----------------------------------------
PRODUCT_NAME = "프리미엄 동원참치액 500g"
PRODUCT_DESC = (
    "동원의 참치액은 훈연참치추출물 80% 이상과 참치엑기스를 사용한 액상 조미료이다. "
    "남태평양 등 청정해역에서 잡은 참치를 5시간 이상 끓여 만든 참치엑기스를 넣어 넣어 깊은 감칠맛을 완성한다. "
    "‘프리미엄 동원참치액’은 훈연참치추출물 85%에 황다랑어 추출물, 사양벌꿀, 다시마, 표고버섯, 감초, 마늘 등을 담아 한층 더 깊고 부드러운 감칠맛을 느낄 수 있다."
)

# -----------------------------------------
# 2) 프롬프트 (대상 제품 한정)
# -----------------------------------------
def build_single_month_prompt(persona_text: str, months: List[str]) -> str:
    months_line = ", ".join(months)
    importance_line = " > ".join(IMPORTANCE_ORDER)
    return f"""
너는 한국 소비자 조미료(참치액) 구매 행동을 분석하는 시장조사 전문가다.

[제품명]
{PRODUCT_NAME}

[제품 설명]
{PRODUCT_DESC}

[페르소나]
{persona_text.strip()}

[질문]
아래 달 중에서 이 페르소나가 위의 제품({PRODUCT_NAME})을 가장 많이 살 것 같은 달 1개만 고르라.
판단 시 아래 속성 중요도(상→하)를 우선 고려하라: {importance_line}

[선택 가능한 달(반드시 이 중에서만 선택)]
{months_line}

[출력 형식(다른 말 절대 금지)]
YYYY-MM
""".strip()

# -----------------------------------------
# 3) 응답 파서
# -----------------------------------------
RE_ISO_LINE  = re.compile(r'^\s*(20\d{2})-(0[1-9]|1[0-2])\s*$', re.MULTILINE)
RE_KOR_LINE  = re.compile(r'^\s*(20\d{2})\s*년\s*(1[0-2]|[1-9])\s*월\s*$', re.MULTILINE)

def parse_one_month(text: str, allowed: List[str]) -> str:
    for y, m in RE_ISO_LINE.findall(text):
        cand = f"{y}-{m}"
        if cand in allowed:
            return cand
    for y, m in RE_KOR_LINE.findall(text):
        cand = f"{y}-{int(m):02d}"
        if cand in allowed:
            return cand
    m = re.search(r'(20\d{2})-(0[1-9]|1[0-2])', text)
    if m:
        cand = f"{m.group(1)}-{m.group(2)}"
        if cand in allowed:
            return cand
    return ""

# -----------------------------------------
# 4) 페르소나 TXT 파싱
# -----------------------------------------
BLOCK_RE = re.compile(
    r"\*\*Persona_(\d{1,3})\*\*\s*(.+?)(?=(?:\n\s*\*\*Persona_\d{1,3}\*\*|\Z))",
    re.DOTALL
)

def parse_personas(text: str) -> List[Tuple[str, str]]:
    personas: List[Tuple[str, str]] = []
    for pid, body in BLOCK_RE.findall(text):
        personas.append((f"Persona_{int(pid):03d}", body.strip()))
    return personas

# -----------------------------------------
# 5) 단일 페르소나 질의
# -----------------------------------------
def ask_month_for_persona(
    pipe,
    persona_text: str,
    temperature: float = 1.20,
    top_p: float = 0.97,
    top_k: int = 0,
    max_new_tokens: int = 12,
    verbose: bool = False
) -> Tuple[str, str]:
    prompt = build_single_month_prompt(persona_text, ALLOWED_MONTHS)
    out = pipe(
        [{"role": "user", "content": prompt}],
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=temperature,
        top_p=top_p,
        top_k=top_k,
        repetition_penalty=1.0,
        return_full_text=False
    )
    raw = out[0].get("generated_text", out[0].get("text", "")).strip()
    month = parse_one_month(raw, ALLOWED_MONTHS) or ""
    if verbose:
        print(f"[RAW] {raw} -> [PARSED] {month or '(fail)'}")
    return month, raw

# -----------------------------------------
# 6) 전체 실행 루프
# -----------------------------------------
def run_survey_over_personas(
    pipe,
    personas_text: str,
    save_dir: str = ".",
    show_progress: bool = True,
    print_details: bool = True
) -> Dict[str, int]:
    os.makedirs(save_dir, exist_ok=True)

    personas = parse_personas(personas_text)
    if not personas:
        raise ValueError("페르소나를 찾지 못했으니 입력 형식을 확인하라.")

    month_counts: Dict[str, int] = {m: 0 for m in ALLOWED_MONTHS}
    total = len(personas)

    log_path = os.path.join(save_dir, "persona_choices.csv")
    with open(log_path, "w", newline="", encoding="utf-8") as f:
        wr = csv.writer(f)
        wr.writerow(["persona_id", "chosen_month", "raw_output"])

        for i, (pid, body) in enumerate(personas, start=1):
            if show_progress:
                print(f"[{i}/{total}] {pid} 질문 중...", flush=True)

            try:
                chosen, raw = ask_month_for_persona(pipe, body, verbose=False)
            except Exception as e:
                chosen, raw = "", f"__ERROR__: {e!r}"

            if chosen:
                month_counts[chosen] += 1
            wr.writerow([pid, chosen, raw])

            if show_progress and print_details:
                short_raw = (raw[:80] + "…") if len(raw) > 80 else raw
                print(f"   ↳ 선택: {chosen or '(파싱 실패)'} | 원문: {short_raw}", flush=True)

    cnt_path = os.path.join(save_dir, "month_counts_premium_500g.csv")
    with open(cnt_path, "w", newline="", encoding="utf-8") as f:
        wr = csv.writer(f)
        wr.writerow(["month", "count"])
        for m in ALLOWED_MONTHS:
            wr.writerow([m, month_counts[m]])

    print("\n[월별 선택 횟수]")
    for m in ALLOWED_MONTHS:
        print(f"{m}: {month_counts[m]}")

    print(f"\n저장 완료:\n- {log_path}\n- {cnt_path}")
    return month_counts

# -----------------------------------------
# 7) 실행
# -----------------------------------------
if __name__ == "__main__":
    month_counts = run_survey_over_personas(
        pipe,
        PERSONAS_RAW,
        save_dir=".",
        show_progress=True,
        print_details=True
    )


In [ ]:
import pandas as pd

# CSV 파일 불러오기
df = pd.read_csv("month_counts_premium_500g.csv", encoding="utf-8")

# 평균 count 계산
mean_count = df["count"].mean()

# factor 및 보정된 adj_factor 계산
df["factor"] = df["count"] / mean_count
df["adj_factor"] = 1 + (df["factor"] -1) / 100

print(df)

# 월별 가중치를 위한 Simulation : 프리미엄 동원참치액 900g

In [ ]:
# -*- coding: utf-8 -*-
import re, csv, os
from typing import List, Tuple, Dict

# -----------------------------------------
# 0) 후보 달 (2024-04 ~ 2025-06)
# -----------------------------------------
ALLOWED_MONTHS: List[str] = [
    "2024-04","2024-05","2024-06","2024-07","2024-08","2024-09",
    "2024-10","2024-11","2024-12","2025-01","2025-02","2025-03",
    "2025-04","2025-05","2025-06"
]

# -----------------------------------------
# 1) 속성 중요도
# -----------------------------------------
IMPORTANCE_ORDER = [
    "최근 1주 외식·배달 횟수", "월평균 가구 소득", "선호 음식 카테고리",
    "요리 여부·빈도", "가구원수", "선호 참치액 브랜드",
    "구매 채널", "자녀 유무", "나이", "성별"
]

# -----------------------------------------
# 1-1) 대상 제품/설명
# -----------------------------------------
PRODUCT_NAME = "프리미엄 동원참치액 900g"
PRODUCT_DESC = (
    "동원의 참치액은 훈연참치추출물 80% 이상과 참치엑기스를 사용한 액상 조미료이다. "
    "남태평양 등 청정해역에서 잡은 참치를 5시간 이상 끓여 만든 참치엑기스를 넣어 넣어 깊은 감칠맛을 완성한다. "
    "‘프리미엄 동원참치액’은 훈연참치추출물 85%에 황다랑어 추출물, 사양벌꿀, 다시마, 표고버섯, 감초, 마늘 등을 담아 한층 더 깊고 부드러운 감칠맛을 느낄 수 있다."
)

# -----------------------------------------
# 2) 프롬프트 (대상 제품 한정)
# -----------------------------------------
def build_single_month_prompt(persona_text: str, months: List[str]) -> str:
    months_line = ", ".join(months)
    importance_line = " > ".join(IMPORTANCE_ORDER)
    return f"""
너는 한국 소비자 조미료(참치액) 구매 행동을 분석하는 시장조사 전문가다.

[제품명]
{PRODUCT_NAME}

[제품 설명]
{PRODUCT_DESC}

[페르소나]
{persona_text.strip()}

[질문]
아래 달 중에서 이 페르소나가 위의 제품({PRODUCT_NAME})을 가장 많이 살 것 같은 달 1개만 고르라.
판단 시 아래 속성 중요도(상→하)를 우선 고려하라: {importance_line}

[선택 가능한 달(반드시 이 중에서만 선택)]
{months_line}

[출력 형식(다른 말 절대 금지)]
YYYY-MM
""".strip()

# -----------------------------------------
# 3) 응답 파서
# -----------------------------------------
RE_ISO_LINE  = re.compile(r'^\s*(20\d{2})-(0[1-9]|1[0-2])\s*$', re.MULTILINE)
RE_KOR_LINE  = re.compile(r'^\s*(20\d{2})\s*년\s*(1[0-2]|[1-9])\s*월\s*$', re.MULTILINE)

def parse_one_month(text: str, allowed: List[str]) -> str:
    for y, m in RE_ISO_LINE.findall(text):
        cand = f"{y}-{m}"
        if cand in allowed:
            return cand
    for y, m in RE_KOR_LINE.findall(text):
        cand = f"{y}-{int(m):02d}"
        if cand in allowed:
            return cand
    m = re.search(r'(20\d{2})-(0[1-9]|1[0-2])', text)
    if m:
        cand = f"{m.group(1)}-{m.group(2)}"
        if cand in allowed:
            return cand
    return ""

# -----------------------------------------
# 4) 페르소나 TXT 파싱
# -----------------------------------------
BLOCK_RE = re.compile(
    r"\*\*Persona_(\d{1,3})\*\*\s*(.+?)(?=(?:\n\s*\*\*Persona_\d{1,3}\*\*|\Z))",
    re.DOTALL
)

def parse_personas(text: str) -> List[Tuple[str, str]]:
    personas: List[Tuple[str, str]] = []
    for pid, body in BLOCK_RE.findall(text):
        personas.append((f"Persona_{int(pid):03d}", body.strip()))
    return personas

# -----------------------------------------
# 5) 단일 페르소나 질의
# -----------------------------------------
def ask_month_for_persona(
    pipe,
    persona_text: str,
    temperature: float = 1.20,
    top_p: float = 0.97,
    top_k: int = 0,
    max_new_tokens: int = 12,
    verbose: bool = False
) -> Tuple[str, str]:
    prompt = build_single_month_prompt(persona_text, ALLOWED_MONTHS)
    out = pipe(
        [{"role": "user", "content": prompt}],
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=temperature,
        top_p=top_p,
        top_k=top_k,
        repetition_penalty=1.0,
        return_full_text=False
    )
    raw = out[0].get("generated_text", out[0].get("text", "")).strip()
    month = parse_one_month(raw, ALLOWED_MONTHS) or ""
    if verbose:
        print(f"[RAW] {raw} -> [PARSED] {month or '(fail)'}")
    return month, raw

# -----------------------------------------
# 6) 전체 실행 루프
# -----------------------------------------
def run_survey_over_personas(
    pipe,
    personas_text: str,
    save_dir: str = ".",
    show_progress: bool = True,
    print_details: bool = True
) -> Dict[str, int]:
    os.makedirs(save_dir, exist_ok=True)

    personas = parse_personas(personas_text)
    if not personas:
        raise ValueError("페르소나를 찾지 못했으니 입력 형식을 확인하라.")

    month_counts: Dict[str, int] = {m: 0 for m in ALLOWED_MONTHS}
    total = len(personas)

    log_path = os.path.join(save_dir, "persona_choices.csv")
    with open(log_path, "w", newline="", encoding="utf-8") as f:
        wr = csv.writer(f)
        wr.writerow(["persona_id", "chosen_month", "raw_output"])

        for i, (pid, body) in enumerate(personas, start=1):
            if show_progress:
                print(f"[{i}/{total}] {pid} 질문 중...", flush=True)

            try:
                chosen, raw = ask_month_for_persona(pipe, body, verbose=False)
            except Exception as e:
                chosen, raw = "", f"__ERROR__: {e!r}"

            if chosen:
                month_counts[chosen] += 1
            wr.writerow([pid, chosen, raw])

            if show_progress and print_details:
                short_raw = (raw[:80] + "…") if len(raw) > 80 else raw
                print(f"   ↳ 선택: {chosen or '(파싱 실패)'} | 원문: {short_raw}", flush=True)

    cnt_path = os.path.join(save_dir, "month_counts_premium_900g.csv")
    with open(cnt_path, "w", newline="", encoding="utf-8") as f:
        wr = csv.writer(f)
        wr.writerow(["month", "count"])
        for m in ALLOWED_MONTHS:
            wr.writerow([m, month_counts[m]])

    print("\n[월별 선택 횟수]")
    for m in ALLOWED_MONTHS:
        print(f"{m}: {month_counts[m]}")

    print(f"\n저장 완료:\n- {log_path}\n- {cnt_path}")
    return month_counts

# -----------------------------------------
# 7) 실행
# -----------------------------------------
if __name__ == "__main__":
    month_counts = run_survey_over_personas(
        pipe,
        PERSONAS_RAW,
        save_dir=".",
        show_progress=True,
        print_details=True
    )


In [ ]:
import pandas as pd

# CSV 파일 불러오기
df = pd.read_csv("month_counts_premium_900g.csv", encoding="utf-8")

# 평균 count 계산
mean_count = df["count"].mean()

# factor 및 보정된 adj_factor 계산
df["factor"] = df["count"] / mean_count
df["adj_factor"] = 1 + (df["factor"] -1) / 100

print(df)

# **라떼**

> # 생성한 라떼 페르소나 불러오기

In [ ]:
import re, csv, os
from typing import List, Tuple

with open("Latte.txt", "r", encoding="utf-8") as f:
    PERSONAS_RAW = f.read()

print("\n".join(PERSONAS_RAW.splitlines()[:5]))

# 속성별 가중치 계산 : 순위로

In [ ]:
# -*- coding: utf-8 -*-
import re
from typing import List, Dict, Tuple

# -----------------------------------------
# 0) 속성 정의 (라떼 페르소나용)
# -----------------------------------------
ATTRS: Dict[int, Tuple[str, str]] = {
    1: ("나이", "10대 / 20대 / 30대 / 40-50대 / 60대 이상"),
    2: ("성별", "남성 / 여성"),
    3: ("가구원수", "혼자 거주하는 1인 가구 / 부부가 함께 사는 2인 가구 / 부모와 자녀 세대가 함께 사는 2세대 이상 가구"),
    4: ("자녀 유무", "있음 / 없음"),
    5: ("월평균 가구 소득 수준", "월평균 가구 소득이 200만원 이상 350만원 미만인 저소득층 / 월평균 가구 소득이 350만원 이상 700만원 이하인 중간 소득층 / 월평균 가구 소득이 700만원을 초과하는 고소득층"),
    6: ("구매 채널", "대형마트와 같은 오프라인 / 쿠팡·마켓컬리·네이버와 같은 온라인 몰 / 온라인+오프라인"),
    7: ("선호하는 라떼 맛/유형", "오리지널(카페라떼) / 바닐라 / 카라멜·헤이즐넛 / 기타(말차·초코 등)"),
    8: ("음용 빈도", "거의 안 마심(월 1회 이하) / 가끔(주 1-2회) / 자주(주 3회 이상)"),
    9: ("우유 종류", "일반 우유 / 저유당·락토프리"),
    10: ("선호하는 라떼 브랜드", "덴마크 우유(동원) / 빙그레 / 매일유업 / 서울우유 브랜드"),
}

# -----------------------------------------
# 옵션: 라운드별 로그 출력 여부 (기본 False)
# -----------------------------------------
VERBOSE = False

# -----------------------------------------
# 1) 프롬프트 생성기
#    - 모델 출력: 반드시 "속성n" (예: 속성10)
# -----------------------------------------
def build_single_pick_prompt(remaining_ids: List[int]) -> str:
    lines = []
    for i in remaining_ids:
        short, detail = ATTRS[i]
        lines.append(f"{i}. {short} — {detail}")
    attr_block = "\n".join(lines)

    prompt = f"""
너는 라떼 구매/음용 행동을 분석하는 시장조사 전문가다.
아래 속성 후보들 중에서 '라떼 구매/음용'에 가장 큰 영향을 미치는 속성 하나만 고르라.

[후보 속성]
{attr_block}

[출력 형식(다른 말 절대 금지)]
속성[번호]

예시:
속성10
""".strip()
    return prompt

# -----------------------------------------
# 2) 파서: "속성n" 한 줄만 허용
#    - (안전장치로 숫자만 응답해도 인식)
# -----------------------------------------
PAT_PROP = re.compile(r'^\s*속성\s*(\d{1,2})\s*$', re.IGNORECASE)
PAT_NUM  = re.compile(r'^\s*(\d{1,2})\s*$')

def parse_single_pick(text: str, allowed: List[int]) -> int:
    txt = text.strip()
    m = PAT_PROP.match(txt)
    if m:
        n = int(m.group(1))
        return n if n in allowed else -1
    m2 = PAT_NUM.match(txt)  # 예외적으로 숫자만 줄 경우
    if m2:
        n = int(m2.group(1))
        return n if n in allowed else -1
    return -1

# -----------------------------------------
# 3) 제거식 랭킹
# -----------------------------------------
def get_latte_importance_elimination(pipe) -> List[int]:
    remaining = list(range(1, 11))
    ranking = []

    round_idx = 1
    while remaining:
        prompt = build_single_pick_prompt(remaining)
        out = pipe(
            [{"role": "user", "content": prompt}],
            max_new_tokens=8,          # "속성n"만 받으면 충분
            do_sample=False,           # 결정적 응답
            return_full_text=False
        )
        raw = out[0].get("generated_text", out[0].get("text", "")).strip()
        picked = parse_single_pick(raw, remaining)

        # 파싱 실패 시 보수적 폴백
        if picked == -1:
            picked = remaining[0]
            if VERBOSE:
                print(f"[라운드 {round_idx}] 파싱 실패 → 폴백 사용, 선택: 속성{picked}")

        if VERBOSE:
            print(f"[라운드 {round_idx}] 모델 출력: {raw}  → 선택: 속성{picked}")

        ranking.append(picked)
        remaining.remove(picked)
        round_idx += 1

    # 최종 요약만 출력
    print("\n[속성별 중요도 순위]  (1위 → 10위)")
    for pos, idx in enumerate(ranking, start=1):
        short, detail = ATTRS[idx]
        print(f"{pos}위: 속성{idx} — {short}")
    return ranking

# -----------------------------------------
# 4) 실행부 (이미 pipe 로드됨)
# -----------------------------------------
if __name__ == "__main__":
    ranking_elim = get_latte_importance_elimination(pipe)

# 월별 가중치를 위한 Simulation : 소화가 잘되는 우유로 만든 카페라떼 250mL

In [ ]:
# -*- coding: utf-8 -*-
import re, csv, os
from typing import List, Tuple, Dict

# -----------------------------------------
# 0) 후보 달 (2024-04 ~ 2025-06)
# -----------------------------------------
ALLOWED_MONTHS: List[str] = [
    "2024-04","2024-05","2024-06","2024-07","2024-08","2024-09",
    "2024-10","2024-11","2024-12","2025-01","2025-02","2025-03",
    "2025-04","2025-05","2025-06"
]

# -----------------------------------------
# 1) 속성 중요도
# -----------------------------------------
IMPORTANCE_ORDER = [
    "선호하는 라떼 맛/유형",       # 속성7
    "월평균 가구 소득 수준",       # 속성5
    "음용 빈도",                  # 속성8
    "선호하는 라떼 브랜드",       # 속성10
    "우유 종류",                  # 속성9
    "가구원수",                   # 속성3
    "구매 채널",                  # 속성6
    "자녀 유무",                  # 속성4
    "나이",                       # 속성1
    "성별"                        # 속성2
]

# -----------------------------------------
# 1-1) 대상 제품/설명
# -----------------------------------------
PRODUCT_NAME = "소화가 잘되는 우유로 만든 카페라떼 250mL"
PRODUCT_DESC = (
    "‘소화가 잘되는 우유로 만든 카페라떼 250mL’는 락토프리(Lactose-Free) 제품으로, "
    "유당을 제거하여 소화 불편 없이 편안하게 즐길 수 있는 프리미엄 카페 음료이다. "
    "저온효소 처리 기술을 적용해 영양소와 풍미를 지키며, 신선한 1등급 원유를 사용해 부드럽고 크리미한 맛을 완성한다. "
    "유당 ZERO로 저당 트렌드를 충족시키면서도 자연스러운 단맛이 살아 있어 건강하게 즐길 수 있다. "
    "특히 2025년 2~6월 사이 SNS를 중심으로 소비자들 사이에서 자발적으로 바이럴되었으며, 별도의 광고 없이 입소문만으로 인기를 얻은 제품이다. "
    "무엇보다 오리지널 라떼 특유의 부드럽고 깔끔한 맛이 특징으로, 기본적인 커피 라떼를 선호하는 소비자들에게 적합하다."
)


# -----------------------------------------
# 2) 프롬프트 (대상 제품 한정)
# -----------------------------------------
def build_single_month_prompt(persona_text: str, months: List[str]) -> str:
    months_line = ", ".join(months)
    importance_line = " > ".join(IMPORTANCE_ORDER)
    return f"""
너는 한국 소비자 라떼 구매·음용 행동을 분석하는 시장조사 전문가다.

[제품명]
{PRODUCT_NAME}

[제품 설명]
{PRODUCT_DESC}

[페르소나]
{persona_text.strip()}

[질문]
아래 달 중에서 이 페르소나가 위의 제품({PRODUCT_NAME})을 가장 많이 살 것 같은 달 1개만 고르라.
판단 시 아래 속성 중요도(상→하)를 우선 고려하라: {importance_line}

[선택 가능한 달(반드시 이 중에서만 선택)]
{months_line}

[출력 형식(다른 말 절대 금지)]
YYYY-MM
""".strip()

# -----------------------------------------
# 3) 응답 파서
# -----------------------------------------
RE_ISO_LINE  = re.compile(r'^\s*(20\d{2})-(0[1-9]|1[0-2])\s*$', re.MULTILINE)
RE_KOR_LINE  = re.compile(r'^\s*(20\d{2})\s*년\s*(1[0-2]|[1-9])\s*월\s*$', re.MULTILINE)

def parse_one_month(text: str, allowed: List[str]) -> str:
    for y, m in RE_ISO_LINE.findall(text):
        cand = f"{y}-{m}"
        if cand in allowed:
            return cand
    for y, m in RE_KOR_LINE.findall(text):
        cand = f"{y}-{int(m):02d}"
        if cand in allowed:
            return cand
    m = re.search(r'(20\d{2})-(0[1-9]|1[0-2])', text)
    if m:
        cand = f"{m.group(1)}-{m.group(2)}"
        if cand in allowed:
            return cand
    return ""

# -----------------------------------------
# 4) 페르소나 TXT 파싱
# -----------------------------------------
BLOCK_RE = re.compile(
    r"\*\*Persona_(\d{1,3})\*\*\s*(.+?)(?=(?:\n\s*\*\*Persona_\d{1,3}\*\*|\Z))",
    re.DOTALL
)

def parse_personas(text: str) -> List[Tuple[str, str]]:
    personas: List[Tuple[str, str]] = []
    for pid, body in BLOCK_RE.findall(text):
        personas.append((f"Persona_{int(pid):03d}", body.strip()))
    return personas

# -----------------------------------------
# 5) 단일 페르소나 질의
# -----------------------------------------
def ask_month_for_persona(
    pipe,
    persona_text: str,
    temperature: float = 1.20,
    top_p: float = 0.97,
    top_k: int = 0,
    max_new_tokens: int = 12,
    verbose: bool = False
) -> Tuple[str, str]:
    prompt = build_single_month_prompt(persona_text, ALLOWED_MONTHS)
    out = pipe(
        [{"role": "user", "content": prompt}],
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=temperature,
        top_p=top_p,
        top_k=top_k,
        repetition_penalty=1.0,
        return_full_text=False
    )
    raw = out[0].get("generated_text", out[0].get("text", "")).strip()
    month = parse_one_month(raw, ALLOWED_MONTHS) or ""
    if verbose:
        print(f"[RAW] {raw} -> [PARSED] {month or '(fail)'}")
    return month, raw

# -----------------------------------------
# 6) 전체 실행 루프
# -----------------------------------------
def run_survey_over_personas(
    pipe,
    personas_text: str,
    save_dir: str = ".",
    show_progress: bool = True,
    print_details: bool = True
) -> Dict[str, int]:
    os.makedirs(save_dir, exist_ok=True)

    personas = parse_personas(personas_text)
    if not personas:
        raise ValueError("페르소나를 찾지 못했으니 입력 형식을 확인하라.")

    month_counts: Dict[str, int] = {m: 0 for m in ALLOWED_MONTHS}
    total = len(personas)

    log_path = os.path.join(save_dir, "persona_choices.csv")
    with open(log_path, "w", newline="", encoding="utf-8") as f:
        wr = csv.writer(f)
        wr.writerow(["persona_id", "chosen_month", "raw_output"])

        for i, (pid, body) in enumerate(personas, start=1):
            if show_progress:
                print(f"[{i}/{total}] {pid} 질문 중...", flush=True)

            try:
                chosen, raw = ask_month_for_persona(pipe, body, verbose=False)
            except Exception as e:
                chosen, raw = "", f"__ERROR__: {e!r}"

            if chosen:
                month_counts[chosen] += 1
            wr.writerow([pid, chosen, raw])

            if show_progress and print_details:
                short_raw = (raw[:80] + "…") if len(raw) > 80 else raw
                print(f"   ↳ 선택: {chosen or '(파싱 실패)'} | 원문: {short_raw}", flush=True)

    cnt_path = os.path.join(save_dir, "month_counts_CafeLatte.csv")
    with open(cnt_path, "w", newline="", encoding="utf-8") as f:
        wr = csv.writer(f)
        wr.writerow(["month", "count"])
        for m in ALLOWED_MONTHS:
            wr.writerow([m, month_counts[m]])

    print("\n[월별 선택 횟수]")
    for m in ALLOWED_MONTHS:
        print(f"{m}: {month_counts[m]}")

    print(f"\n저장 완료:\n- {log_path}\n- {cnt_path}")
    return month_counts

# -----------------------------------------
# 7) 실행
# -----------------------------------------
if __name__ == "__main__":
    month_counts = run_survey_over_personas(
        pipe,
        PERSONAS_RAW,
        save_dir=".",
        show_progress=True,
        print_details=True
    )


In [ ]:
import pandas as pd

# CSV 파일 불러오기
df = pd.read_csv("month_counts_CafeLatte.csv", encoding="utf-8")

# 평균 count 계산
mean_count = df["count"].mean()

# factor 및 보정된 adj_factor 계산
df["factor"] = df["count"] / mean_count
df["adj_factor"] = 1 + (df["factor"] -1) / 100

print(df)

# 월별 가중치를 위한 Simulation : 소화가 잘되는 우유로 만든 바닐라라떼 250mL

In [ ]:
# -*- coding: utf-8 -*-
import re, csv, os
from typing import List, Tuple, Dict

# -----------------------------------------
# 0) 후보 달 (2024-04 ~ 2025-06)
# -----------------------------------------
ALLOWED_MONTHS: List[str] = [
    "2024-04","2024-05","2024-06","2024-07","2024-08","2024-09",
    "2024-10","2024-11","2024-12","2025-01","2025-02","2025-03",
    "2025-04","2025-05","2025-06"
]

# -----------------------------------------
# 1) 속성 중요도
# -----------------------------------------
IMPORTANCE_ORDER = [
    "선호하는 라떼 맛/유형",       # 속성7
    "월평균 가구 소득 수준",       # 속성5
    "음용 빈도",                  # 속성8
    "선호하는 라떼 브랜드",       # 속성10
    "우유 종류",                  # 속성9
    "가구원수",                   # 속성3
    "구매 채널",                  # 속성6
    "자녀 유무",                  # 속성4
    "나이",                       # 속성1
    "성별"                        # 속성2
]

# -----------------------------------------
# 1-1) 대상 제품/설명
# -----------------------------------------
PRODUCT_NAME = "소화가 잘되는 우유로 만든 바닐라라떼 250mL"
PRODUCT_DESC = (
    "‘소화가 잘되는 우유로 만든 바닐라라떼 250mL’는 락토프리(Lactose-Free) 제품으로, "
    "유당을 제거하여 소화 불편 없이 편안하게 즐길 수 있는 프리미엄 카페 음료이다. "
    "저온효소 처리 기술을 적용해 영양소와 풍미를 지키며, 신선한 1등급 원유를 사용해 부드럽고 크리미한 맛을 완성한다. "
    "유당 ZERO로 저당 트렌드를 충족시키면서도 자연스러운 단맛이 살아 있어 건강하게 즐길 수 있다. "
    "특히 2025년 2~6월 사이 SNS를 중심으로 소비자들 사이에서 자발적으로 바이럴되었으며, 별도의 광고 없이 입소문만으로 인기를 얻은 제품이다. "
    "바닐라 특유의 달콤하고 향긋한 풍미가 더해져, 달콤한 라떼를 선호하는 소비자층에게 매력적인 선택지로 자리 잡고 있다."
)


# -----------------------------------------
# 2) 프롬프트 (대상 제품 한정)
# -----------------------------------------
def build_single_month_prompt(persona_text: str, months: List[str]) -> str:
    months_line = ", ".join(months)
    importance_line = " > ".join(IMPORTANCE_ORDER)
    return f"""
너는 한국 소비자 라떼 구매·음용 행동을 분석하는 시장조사 전문가다.

[제품명]
{PRODUCT_NAME}

[제품 설명]
{PRODUCT_DESC}

[페르소나]
{persona_text.strip()}

[질문]
아래 달 중에서 이 페르소나가 위의 제품({PRODUCT_NAME})을 가장 많이 살 것 같은 달 1개만 고르라.
판단 시 아래 속성 중요도(상→하)를 우선 고려하라: {importance_line}

[선택 가능한 달(반드시 이 중에서만 선택)]
{months_line}

[출력 형식(다른 말 절대 금지)]
YYYY-MM
""".strip()

# -----------------------------------------
# 3) 응답 파서
# -----------------------------------------
RE_ISO_LINE  = re.compile(r'^\s*(20\d{2})-(0[1-9]|1[0-2])\s*$', re.MULTILINE)
RE_KOR_LINE  = re.compile(r'^\s*(20\d{2})\s*년\s*(1[0-2]|[1-9])\s*월\s*$', re.MULTILINE)

def parse_one_month(text: str, allowed: List[str]) -> str:
    for y, m in RE_ISO_LINE.findall(text):
        cand = f"{y}-{m}"
        if cand in allowed:
            return cand
    for y, m in RE_KOR_LINE.findall(text):
        cand = f"{y}-{int(m):02d}"
        if cand in allowed:
            return cand
    m = re.search(r'(20\d{2})-(0[1-9]|1[0-2])', text)
    if m:
        cand = f"{m.group(1)}-{m.group(2)}"
        if cand in allowed:
            return cand
    return ""

# -----------------------------------------
# 4) 페르소나 TXT 파싱
# -----------------------------------------
BLOCK_RE = re.compile(
    r"\*\*Persona_(\d{1,3})\*\*\s*(.+?)(?=(?:\n\s*\*\*Persona_\d{1,3}\*\*|\Z))",
    re.DOTALL
)

def parse_personas(text: str) -> List[Tuple[str, str]]:
    personas: List[Tuple[str, str]] = []
    for pid, body in BLOCK_RE.findall(text):
        personas.append((f"Persona_{int(pid):03d}", body.strip()))
    return personas

# -----------------------------------------
# 5) 단일 페르소나 질의
# -----------------------------------------
def ask_month_for_persona(
    pipe,
    persona_text: str,
    temperature: float = 1.20,
    top_p: float = 0.97,
    top_k: int = 0,
    max_new_tokens: int = 12,
    verbose: bool = False
) -> Tuple[str, str]:
    prompt = build_single_month_prompt(persona_text, ALLOWED_MONTHS)
    out = pipe(
        [{"role": "user", "content": prompt}],
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=temperature,
        top_p=top_p,
        top_k=top_k,
        repetition_penalty=1.0,
        return_full_text=False
    )
    raw = out[0].get("generated_text", out[0].get("text", "")).strip()
    month = parse_one_month(raw, ALLOWED_MONTHS) or ""
    if verbose:
        print(f"[RAW] {raw} -> [PARSED] {month or '(fail)'}")
    return month, raw

# -----------------------------------------
# 6) 전체 실행 루프
# -----------------------------------------
def run_survey_over_personas(
    pipe,
    personas_text: str,
    save_dir: str = ".",
    show_progress: bool = True,
    print_details: bool = True
) -> Dict[str, int]:
    os.makedirs(save_dir, exist_ok=True)

    personas = parse_personas(personas_text)
    if not personas:
        raise ValueError("페르소나를 찾지 못했으니 입력 형식을 확인하라.")

    month_counts: Dict[str, int] = {m: 0 for m in ALLOWED_MONTHS}
    total = len(personas)

    log_path = os.path.join(save_dir, "persona_choices.csv")
    with open(log_path, "w", newline="", encoding="utf-8") as f:
        wr = csv.writer(f)
        wr.writerow(["persona_id", "chosen_month", "raw_output"])

        for i, (pid, body) in enumerate(personas, start=1):
            if show_progress:
                print(f"[{i}/{total}] {pid} 질문 중...", flush=True)

            try:
                chosen, raw = ask_month_for_persona(pipe, body, verbose=False)
            except Exception as e:
                chosen, raw = "", f"__ERROR__: {e!r}"

            if chosen:
                month_counts[chosen] += 1
            wr.writerow([pid, chosen, raw])

            if show_progress and print_details:
                short_raw = (raw[:80] + "…") if len(raw) > 80 else raw
                print(f"   ↳ 선택: {chosen or '(파싱 실패)'} | 원문: {short_raw}", flush=True)

    cnt_path = os.path.join(save_dir, "month_counts_VanillaLatte.csv")
    with open(cnt_path, "w", newline="", encoding="utf-8") as f:
        wr = csv.writer(f)
        wr.writerow(["month", "count"])
        for m in ALLOWED_MONTHS:
            wr.writerow([m, month_counts[m]])

    print("\n[월별 선택 횟수]")
    for m in ALLOWED_MONTHS:
        print(f"{m}: {month_counts[m]}")

    print(f"\n저장 완료:\n- {log_path}\n- {cnt_path}")
    return month_counts

# -----------------------------------------
# 7) 실행
# -----------------------------------------
if __name__ == "__main__":
    month_counts = run_survey_over_personas(
        pipe,
        PERSONAS_RAW,
        save_dir=".",
        show_progress=True,
        print_details=True
    )


In [ ]:
import pandas as pd

# CSV 파일 불러오기
df = pd.read_csv("month_counts_VanillaLatte.csv", encoding="utf-8")

# 평균 count 계산
mean_count = df["count"].mean()

# factor 및 보정된 adj_factor 계산
df["factor"] = df["count"] / mean_count
df["adj_factor"] = 1 + (df["factor"] -1) / 100

print(df)

# **동원 맛참 참치캔**

> # 생성한 참치캔 페르소나 불러오기

In [ ]:
import re, csv, os
from typing import List, Tuple

with open("chamchican.txt", "r", encoding="utf-8") as f:
    PERSONAS_RAW = f.read()

print("\n".join(PERSONAS_RAW.splitlines()[:5]))

# 속성별 가중치 계산 : 순위로

In [ ]:
# -*- coding: utf-8 -*-
import re
from typing import List, Dict, Tuple

# -----------------------------------------
# 0) 속성 정의 (참치캔 페르소나용)
# -----------------------------------------
ATTRS: Dict[int, Tuple[str, str]] = {
    1: ("나이", "10대 / 20대 / 30대 / 40-50대 / 60대 이상"),
    2: ("성별", "남성 / 여성"),
    3: ("가구원수", "혼자 거주하는 1인 가구 / 부부가 함께 사는 2인 가구 / 부모와 자녀 세대가 함께 사는 2세대 이상 가구"),
    4: ("자녀 유무", "있음 / 없음"),
    5: ("월평균 가구 소득 수준", "월평균 가구 소득이 200만원 이상 350만원 미만인 저소득층 / 월평균 가구 소득이 350만원 이상 700만원 이하인 중간 소득층 / 월평균 가구 소득이 700만원을 초과하는 고소득층"),
    6: ("구매 채널", "대형마트와 같은 오프라인 / 쿠팡·마켓컬리·네이버와 같은 온라인 몰 / 온라인+오프라인"),
    7: ("참치캔 주 사용 용도", "샐러드·다이어트 / 주먹밥·도시락 / 찌개·볶음"),
    8: ("요리 여부·빈도", "전혀 하지 않음 / 가끔(주 1-2회) / 자주(주 3회 이상)"),
    9: ("선호하는 참치캔 타입", "오일(라이트) / 매운맛(고추·칠리) / 기타"),
    10: ("선호하는 참치캔 브랜드", "동원 / 사조대림 / 오뚜기 / 기타 브랜드"),
}

# -----------------------------------------
# 옵션: 라운드별 로그 출력 여부 (기본 False)
# -----------------------------------------
VERBOSE = False

# -----------------------------------------
# 1) 프롬프트 생성기
#    - 모델 출력: 반드시 "속성n" (예: 속성10)
# -----------------------------------------
def build_single_pick_prompt(remaining_ids: List[int]) -> str:
    lines = []
    for i in remaining_ids:
        short, detail = ATTRS[i]
        lines.append(f"{i}. {short} — {detail}")
    attr_block = "\n".join(lines)

    prompt = f"""
너는 참치캔 구매·섭취 행동을 분석하는 시장조사 전문가다.
아래 속성 후보들 중에서 '참치캔 구매·섭취'에 가장 큰 영향을 미치는 속성 하나만 고르라.

[후보 속성]
{attr_block}

[출력 형식(다른 말 절대 금지)]
속성[번호]

예시:
속성10
""".strip()
    return prompt

# -----------------------------------------
# 2) 파서: "속성n" 한 줄만 허용
# -----------------------------------------
PAT_PROP = re.compile(r'^\s*속성\s*(\d{1,2})\s*$', re.IGNORECASE)
PAT_NUM  = re.compile(r'^\s*(\d{1,2})\s*$')

def parse_single_pick(text: str, allowed: List[int]) -> int:
    txt = text.strip()
    m = PAT_PROP.match(txt)
    if m:
        n = int(m.group(1))
        return n if n in allowed else -1
    m2 = PAT_NUM.match(txt)  # 숫자만 줄 경우 허용
    if m2:
        n = int(m2.group(1))
        return n if n in allowed else -1
    return -1

# -----------------------------------------
# 3) 제거식 랭킹
# -----------------------------------------
def get_tuna_can_importance_elimination(pipe) -> List[int]:
    remaining = list(range(1, 11))
    ranking = []

    round_idx = 1
    while remaining:
        prompt = build_single_pick_prompt(remaining)
        out = pipe(
            [{"role": "user", "content": prompt}],
            max_new_tokens=8,          # "속성n"만 받으면 충분
            do_sample=False,           # 결정적 응답
            return_full_text=False
        )
        raw = out[0].get("generated_text", out[0].get("text", "")).strip()
        picked = parse_single_pick(raw, remaining)

        # 파싱 실패 시 보수적 폴백
        if picked == -1:
            picked = remaining[0]
            if VERBOSE:
                print(f"[라운드 {round_idx}] 파싱 실패 → 폴백 사용, 선택: 속성{picked}")

        if VERBOSE:
            print(f"[라운드 {round_idx}] 모델 출력: {raw}  → 선택: 속성{picked}")

        ranking.append(picked)
        remaining.remove(picked)
        round_idx += 1

    # 최종 요약만 출력
    print("\n[속성별 중요도 순위]  (1위 → 10위)")
    for pos, idx in enumerate(ranking, start=1):
        short, detail = ATTRS[idx]
        print(f"{pos}위: 속성{idx} — {short}")
    return ranking

# -----------------------------------------
# 4) 실행부 (이미 pipe 로드됨)
# -----------------------------------------
if __name__ == "__main__":
    ranking_elim = get_tuna_can_importance_elimination(pipe)

# 월별 가중치를 위한 Simulation : 동원맛참 고소참기름 135g

In [ ]:
# -*- coding: utf-8 -*-
import re, csv, os
from typing import List, Tuple, Dict

# -----------------------------------------
# 0) 후보 달 (2024-04 ~ 2025-06)
# -----------------------------------------
ALLOWED_MONTHS: List[str] = [
    "2024-04","2024-05","2024-06","2024-07","2024-08","2024-09",
    "2024-10","2024-11","2024-12","2025-01","2025-02","2025-03",
    "2025-04","2025-05","2025-06"
]

# -----------------------------------------
# 1) 속성 중요도
# -----------------------------------------
IMPORTANCE_ORDER = [
    "요리 여부·빈도",             # 속성8
    "월평균 가구 소득 수준",       # 속성5
    "참치캔 주 사용 용도",         # 속성7
    "가구원수",                   # 속성3
    "자녀 유무",                   # 속성4
    "선호하는 참치캔 브랜드",      # 속성10
    "구매 채널",                   # 속성6
    "성별",                        # 속성2
    "선호하는 참치캔 타입",        # 속성9
    "나이"                         # 속성1
]

# -----------------------------------------
# 1-1) 대상 제품/설명
# -----------------------------------------
PRODUCT_NAME = "동원맛참 고소참기름 135g"
PRODUCT_DESC = (
    "‘동원맛참 고소참기름’은 선별한 고품질 참깨를 저온에서 압착해 참깨 본연의 고소하고 깊은 풍미를 살린 프리미엄 참기름 제품이다. "
    "단백질과 필수 미네랄인 셀레늄을 함유해 맛뿐 아니라 건강까지 고려할 수 있는 점이 특징이다. "
    "잡내 없이 깔끔하면서도 진한 고소함을 구현해, 나물·비빔밥·볶음 요리 등 다양한 한식 메뉴에 폭넓게 활용된다. "
    "소용량부터 가정용 대용량까지 다양한 용량으로 출시되어 1~2인 가구부터 다인 가구까지 폭넓은 소비자층을 타깃으로 한다. "
    "2025년 상반기에는 TV, YouTube, SNS를 아우르는 통합 마케팅을 전개했으며, 특히 수도권 주요 오프라인 매장에서 시식 행사와 체험 프로모션을 병행해 소비자 경험을 강화하였다. "
    "광고 모델로는 그룹 아이브(IVE)의 안유진을 기용하여 MZ세대에게 세련되고 건강한 이미지를 각인시킨 점이 돋보인다. "
    "맛과 영양, 브랜드 신뢰도를 동시에 잡아 참기름 시장에서 차별화된 포지셔닝을 구축한 제품이다."
)



# -----------------------------------------
# 2) 프롬프트 (대상 제품 한정)
# -----------------------------------------
def build_single_month_prompt(persona_text: str, months: List[str]) -> str:
    months_line = ", ".join(months)
    importance_line = " > ".join(IMPORTANCE_ORDER)
    return f"""
너는 한국 소비자 그릭요거트 구매·섭취 행동을 분석하는 시장조사 전문가다.

[제품명]
{PRODUCT_NAME}

[제품 설명]
{PRODUCT_DESC}

[페르소나]
{persona_text.strip()}

[질문]
아래 달 중에서 이 페르소나가 위의 제품({PRODUCT_NAME})을 가장 많이 살 것 같은 달 1개만 고르라.
판단 시 아래 속성 중요도(상→하)를 우선 고려하라: {importance_line}

[선택 가능한 달(반드시 이 중에서만 선택)]
{months_line}

[출력 형식(다른 말 절대 금지)]
YYYY-MM
""".strip()

# -----------------------------------------
# 3) 응답 파서
# -----------------------------------------
RE_ISO_LINE  = re.compile(r'^\s*(20\d{2})-(0[1-9]|1[0-2])\s*$', re.MULTILINE)
RE_KOR_LINE  = re.compile(r'^\s*(20\d{2})\s*년\s*(1[0-2]|[1-9])\s*월\s*$', re.MULTILINE)

def parse_one_month(text: str, allowed: List[str]) -> str:
    for y, m in RE_ISO_LINE.findall(text):
        cand = f"{y}-{m}"
        if cand in allowed:
            return cand
    for y, m in RE_KOR_LINE.findall(text):
        cand = f"{y}-{int(m):02d}"
        if cand in allowed:
            return cand
    m = re.search(r'(20\d{2})-(0[1-9]|1[0-2])', text)
    if m:
        cand = f"{m.group(1)}-{m.group(2)}"
        if cand in allowed:
            return cand
    return ""

# -----------------------------------------
# 4) 페르소나 TXT 파싱
# -----------------------------------------
BLOCK_RE = re.compile(
    r"\*\*Persona_(\d{1,3})\*\*\s*(.+?)(?=(?:\n\s*\*\*Persona_\d{1,3}\*\*|\Z))",
    re.DOTALL
)

def parse_personas(text: str) -> List[Tuple[str, str]]:
    personas: List[Tuple[str, str]] = []
    for pid, body in BLOCK_RE.findall(text):
        personas.append((f"Persona_{int(pid):03d}", body.strip()))
    return personas

# -----------------------------------------
# 5) 단일 페르소나 질의
# -----------------------------------------
def ask_month_for_persona(
    pipe,
    persona_text: str,
    temperature: float = 1.20,
    top_p: float = 0.97,
    top_k: int = 0,
    max_new_tokens: int = 12,
    verbose: bool = False
) -> Tuple[str, str]:
    prompt = build_single_month_prompt(persona_text, ALLOWED_MONTHS)
    out = pipe(
        [{"role": "user", "content": prompt}],
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=temperature,
        top_p=top_p,
        top_k=top_k,
        repetition_penalty=1.0,
        return_full_text=False
    )
    raw = out[0].get("generated_text", out[0].get("text", "")).strip()
    month = parse_one_month(raw, ALLOWED_MONTHS) or ""
    if verbose:
        print(f"[RAW] {raw} -> [PARSED] {month or '(fail)'}")
    return month, raw

# -----------------------------------------
# 6) 전체 실행 루프
# -----------------------------------------
def run_survey_over_personas(
    pipe,
    personas_text: str,
    save_dir: str = ".",
    show_progress: bool = True,
    print_details: bool = True
) -> Dict[str, int]:
    os.makedirs(save_dir, exist_ok=True)

    personas = parse_personas(personas_text)
    if not personas:
        raise ValueError("페르소나를 찾지 못했으니 입력 형식을 확인하라.")

    month_counts: Dict[str, int] = {m: 0 for m in ALLOWED_MONTHS}
    total = len(personas)

    log_path = os.path.join(save_dir, "persona_choices.csv")
    with open(log_path, "w", newline="", encoding="utf-8") as f:
        wr = csv.writer(f)
        wr.writerow(["persona_id", "chosen_month", "raw_output"])

        for i, (pid, body) in enumerate(personas, start=1):
            if show_progress:
                print(f"[{i}/{total}] {pid} 질문 중...", flush=True)

            try:
                chosen, raw = ask_month_for_persona(pipe, body, verbose=False)
            except Exception as e:
                chosen, raw = "", f"__ERROR__: {e!r}"

            if chosen:
                month_counts[chosen] += 1
            wr.writerow([pid, chosen, raw])

            if show_progress and print_details:
                short_raw = (raw[:80] + "…") if len(raw) > 80 else raw
                print(f"   ↳ 선택: {chosen or '(파싱 실패)'} | 원문: {short_raw}", flush=True)

    cnt_path = os.path.join(save_dir, "month_counts_Goso_Oil_135g.csv")
    with open(cnt_path, "w", newline="", encoding="utf-8") as f:
        wr = csv.writer(f)
        wr.writerow(["month", "count"])
        for m in ALLOWED_MONTHS:
            wr.writerow([m, month_counts[m]])

    print("\n[월별 선택 횟수]")
    for m in ALLOWED_MONTHS:
        print(f"{m}: {month_counts[m]}")

    print(f"\n저장 완료:\n- {log_path}\n- {cnt_path}")
    return month_counts

# -----------------------------------------
# 7) 실행
# -----------------------------------------
if __name__ == "__main__":
    month_counts = run_survey_over_personas(
        pipe,
        PERSONAS_RAW,
        save_dir=".",
        show_progress=True,
        print_details=True
    )


In [ ]:
import pandas as pd

# CSV 파일 불러오기
df = pd.read_csv("month_counts_Goso_Oil_135g.csv", encoding="utf-8")

# 평균 count 계산
mean_count = df["count"].mean()

# factor 및 보정된 adj_factor 계산
df["factor"] = df["count"] / mean_count
df["adj_factor"] = 1 + (df["factor"] -1) / 100

print(df)

# 월별 가중치를 위한 Simulation : 동원맛참 고소참기름 90g

In [ ]:
# -*- coding: utf-8 -*-
import re, csv, os
from typing import List, Tuple, Dict

# -----------------------------------------
# 0) 후보 달 (2024-04 ~ 2025-06)
# -----------------------------------------
ALLOWED_MONTHS: List[str] = [
    "2024-04","2024-05","2024-06","2024-07","2024-08","2024-09",
    "2024-10","2024-11","2024-12","2025-01","2025-02","2025-03",
    "2025-04","2025-05","2025-06"
]

# -----------------------------------------
# 1) 속성 중요도
# -----------------------------------------
IMPORTANCE_ORDER = [
    "요리 여부·빈도",             # 속성8
    "월평균 가구 소득 수준",       # 속성5
    "참치캔 주 사용 용도",         # 속성7
    "가구원수",                   # 속성3
    "자녀 유무",                   # 속성4
    "선호하는 참치캔 브랜드",      # 속성10
    "구매 채널",                   # 속성6
    "성별",                        # 속성2
    "선호하는 참치캔 타입",        # 속성9
    "나이"                         # 속성1
]

# -----------------------------------------
# 1-1) 대상 제품/설명
# -----------------------------------------
PRODUCT_NAME = "동원맛참 고소참기름 90g"
PRODUCT_DESC = (
    "‘동원맛참 고소참기름’은 선별한 고품질 참깨를 저온에서 압착해 참깨 본연의 고소하고 깊은 풍미를 살린 프리미엄 참기름 제품이다. "
    "단백질과 필수 미네랄인 셀레늄을 함유해 맛뿐 아니라 건강까지 고려할 수 있는 점이 특징이다. "
    "잡내 없이 깔끔하면서도 진한 고소함을 구현해, 나물·비빔밥·볶음 요리 등 다양한 한식 메뉴에 폭넓게 활용된다. "
    "소용량부터 가정용 대용량까지 다양한 용량으로 출시되어 1~2인 가구부터 다인 가구까지 폭넓은 소비자층을 타깃으로 한다. "
    "2025년 상반기에는 TV, YouTube, SNS를 아우르는 통합 마케팅을 전개했으며, 특히 수도권 주요 오프라인 매장에서 시식 행사와 체험 프로모션을 병행해 소비자 경험을 강화하였다. "
    "광고 모델로는 그룹 아이브(IVE)의 안유진을 기용하여 MZ세대에게 세련되고 건강한 이미지를 각인시킨 점이 돋보인다. "
    "맛과 영양, 브랜드 신뢰도를 동시에 잡아 참기름 시장에서 차별화된 포지셔닝을 구축한 제품이다."
)



# -----------------------------------------
# 2) 프롬프트 (대상 제품 한정)
# -----------------------------------------
def build_single_month_prompt(persona_text: str, months: List[str]) -> str:
    months_line = ", ".join(months)
    importance_line = " > ".join(IMPORTANCE_ORDER)
    return f"""
너는 한국 소비자 그릭요거트 구매·섭취 행동을 분석하는 시장조사 전문가다.

[제품명]
{PRODUCT_NAME}

[제품 설명]
{PRODUCT_DESC}

[페르소나]
{persona_text.strip()}

[질문]
아래 달 중에서 이 페르소나가 위의 제품({PRODUCT_NAME})을 가장 많이 살 것 같은 달 1개만 고르라.
판단 시 아래 속성 중요도(상→하)를 우선 고려하라: {importance_line}

[선택 가능한 달(반드시 이 중에서만 선택)]
{months_line}

[출력 형식(다른 말 절대 금지)]
YYYY-MM
""".strip()

# -----------------------------------------
# 3) 응답 파서
# -----------------------------------------
RE_ISO_LINE  = re.compile(r'^\s*(20\d{2})-(0[1-9]|1[0-2])\s*$', re.MULTILINE)
RE_KOR_LINE  = re.compile(r'^\s*(20\d{2})\s*년\s*(1[0-2]|[1-9])\s*월\s*$', re.MULTILINE)

def parse_one_month(text: str, allowed: List[str]) -> str:
    for y, m in RE_ISO_LINE.findall(text):
        cand = f"{y}-{m}"
        if cand in allowed:
            return cand
    for y, m in RE_KOR_LINE.findall(text):
        cand = f"{y}-{int(m):02d}"
        if cand in allowed:
            return cand
    m = re.search(r'(20\d{2})-(0[1-9]|1[0-2])', text)
    if m:
        cand = f"{m.group(1)}-{m.group(2)}"
        if cand in allowed:
            return cand
    return ""

# -----------------------------------------
# 4) 페르소나 TXT 파싱
# -----------------------------------------
BLOCK_RE = re.compile(
    r"\*\*Persona_(\d{1,3})\*\*\s*(.+?)(?=(?:\n\s*\*\*Persona_\d{1,3}\*\*|\Z))",
    re.DOTALL
)

def parse_personas(text: str) -> List[Tuple[str, str]]:
    personas: List[Tuple[str, str]] = []
    for pid, body in BLOCK_RE.findall(text):
        personas.append((f"Persona_{int(pid):03d}", body.strip()))
    return personas

# -----------------------------------------
# 5) 단일 페르소나 질의
# -----------------------------------------
def ask_month_for_persona(
    pipe,
    persona_text: str,
    temperature: float = 1.20,
    top_p: float = 0.97,
    top_k: int = 0,
    max_new_tokens: int = 12,
    verbose: bool = False
) -> Tuple[str, str]:
    prompt = build_single_month_prompt(persona_text, ALLOWED_MONTHS)
    out = pipe(
        [{"role": "user", "content": prompt}],
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=temperature,
        top_p=top_p,
        top_k=top_k,
        repetition_penalty=1.0,
        return_full_text=False
    )
    raw = out[0].get("generated_text", out[0].get("text", "")).strip()
    month = parse_one_month(raw, ALLOWED_MONTHS) or ""
    if verbose:
        print(f"[RAW] {raw} -> [PARSED] {month or '(fail)'}")
    return month, raw

# -----------------------------------------
# 6) 전체 실행 루프
# -----------------------------------------
def run_survey_over_personas(
    pipe,
    personas_text: str,
    save_dir: str = ".",
    show_progress: bool = True,
    print_details: bool = True
) -> Dict[str, int]:
    os.makedirs(save_dir, exist_ok=True)

    personas = parse_personas(personas_text)
    if not personas:
        raise ValueError("페르소나를 찾지 못했으니 입력 형식을 확인하라.")

    month_counts: Dict[str, int] = {m: 0 for m in ALLOWED_MONTHS}
    total = len(personas)

    log_path = os.path.join(save_dir, "persona_choices.csv")
    with open(log_path, "w", newline="", encoding="utf-8") as f:
        wr = csv.writer(f)
        wr.writerow(["persona_id", "chosen_month", "raw_output"])

        for i, (pid, body) in enumerate(personas, start=1):
            if show_progress:
                print(f"[{i}/{total}] {pid} 질문 중...", flush=True)

            try:
                chosen, raw = ask_month_for_persona(pipe, body, verbose=False)
            except Exception as e:
                chosen, raw = "", f"__ERROR__: {e!r}"

            if chosen:
                month_counts[chosen] += 1
            wr.writerow([pid, chosen, raw])

            if show_progress and print_details:
                short_raw = (raw[:80] + "…") if len(raw) > 80 else raw
                print(f"   ↳ 선택: {chosen or '(파싱 실패)'} | 원문: {short_raw}", flush=True)

    cnt_path = os.path.join(save_dir, "month_counts_Goso_Oil_90g.csv")
    with open(cnt_path, "w", newline="", encoding="utf-8") as f:
        wr = csv.writer(f)
        wr.writerow(["month", "count"])
        for m in ALLOWED_MONTHS:
            wr.writerow([m, month_counts[m]])

    print("\n[월별 선택 횟수]")
    for m in ALLOWED_MONTHS:
        print(f"{m}: {month_counts[m]}")

    print(f"\n저장 완료:\n- {log_path}\n- {cnt_path}")
    return month_counts

# -----------------------------------------
# 7) 실행
# -----------------------------------------
if __name__ == "__main__":
    month_counts = run_survey_over_personas(
        pipe,
        PERSONAS_RAW,
        save_dir=".",
        show_progress=True,
        print_details=True
    )


In [ ]:
import pandas as pd

# CSV 파일 불러오기
df = pd.read_csv("month_counts_Goso_Oil_90g.csv", encoding="utf-8")

# 평균 count 계산
mean_count = df["count"].mean()

# factor 및 보정된 adj_factor 계산
df["factor"] = df["count"] / mean_count
df["adj_factor"] = 1 + (df["factor"] -1) / 100

print(df)

# 월별 가중치를 위한 Simulation : 동원맛참 매콤참기름 135g

In [ ]:
# -*- coding: utf-8 -*-
import re, csv, os
from typing import List, Tuple, Dict

# -----------------------------------------
# 0) 후보 달 (2024-04 ~ 2025-06)
# -----------------------------------------
ALLOWED_MONTHS: List[str] = [
    "2024-04","2024-05","2024-06","2024-07","2024-08","2024-09",
    "2024-10","2024-11","2024-12","2025-01","2025-02","2025-03",
    "2025-04","2025-05","2025-06"
]

# -----------------------------------------
# 1) 속성 중요도
# -----------------------------------------
IMPORTANCE_ORDER = [
    "요리 여부·빈도",             # 속성8
    "월평균 가구 소득 수준",       # 속성5
    "참치캔 주 사용 용도",         # 속성7
    "가구원수",                   # 속성3
    "자녀 유무",                   # 속성4
    "선호하는 참치캔 브랜드",      # 속성10
    "구매 채널",                   # 속성6
    "성별",                        # 속성2
    "선호하는 참치캔 타입",        # 속성9
    "나이"                         # 속성1
]

# -----------------------------------------
# 1-1) 대상 제품/설명
# -----------------------------------------
PRODUCT_NAME = "동원맛참 매콤참기름 135g"
PRODUCT_DESC = (
    "‘동원맛참 매콤참기름’은 고소한 참깨 본연의 풍미에 청양고추와 고추기름을 더해 매콤한 맛을 구현한 프리미엄 참기름 제품이다. "
    "단백질과 필수 미네랄인 셀레늄을 함유해 맛뿐 아니라 건강까지 고려할 수 있으며, "
    "기존 참기름의 고소함과 함께 매콤한 풍미가 어우러져 비빔밥·국수·볶음·무침 등 다양한 한식 요리에 색다른 매력을 더한다. "
    "캡사이신 대신 실제 고추를 우려내어 자극적이지 않고 깔끔하게 오래 남는 매운맛이 특징이다. "
    "2025년 상반기에는 ‘한 방울로 매콤한 변화’를 콘셉트로 TV, YouTube, SNS를 아우르는 마케팅 캠페인을 전개했으며, "
    "수도권 대형마트에서는 시식 행사와 쿠킹 클래스 협업으로 소비자 경험을 강화하였다. "
    "광고 모델로는 그룹 아이브(IVE)의 안유진을 기용해 MZ세대에게 세련되고 건강한 이미지를 각인시켰으며, "
    "참기름의 고소함과 매콤함을 결합한 새로운 카테고리 제품으로 자리매김하고 있다."
)

# -----------------------------------------
# 2) 프롬프트 (대상 제품 한정)
# -----------------------------------------
def build_single_month_prompt(persona_text: str, months: List[str]) -> str:
    months_line = ", ".join(months)
    importance_line = " > ".join(IMPORTANCE_ORDER)
    return f"""
너는 한국 소비자 그릭요거트 구매·섭취 행동을 분석하는 시장조사 전문가다.

[제품명]
{PRODUCT_NAME}

[제품 설명]
{PRODUCT_DESC}

[페르소나]
{persona_text.strip()}

[질문]
아래 달 중에서 이 페르소나가 위의 제품({PRODUCT_NAME})을 가장 많이 살 것 같은 달 1개만 고르라.
판단 시 아래 속성 중요도(상→하)를 우선 고려하라: {importance_line}

[선택 가능한 달(반드시 이 중에서만 선택)]
{months_line}

[출력 형식(다른 말 절대 금지)]
YYYY-MM
""".strip()

# -----------------------------------------
# 3) 응답 파서
# -----------------------------------------
RE_ISO_LINE  = re.compile(r'^\s*(20\d{2})-(0[1-9]|1[0-2])\s*$', re.MULTILINE)
RE_KOR_LINE  = re.compile(r'^\s*(20\d{2})\s*년\s*(1[0-2]|[1-9])\s*월\s*$', re.MULTILINE)

def parse_one_month(text: str, allowed: List[str]) -> str:
    for y, m in RE_ISO_LINE.findall(text):
        cand = f"{y}-{m}"
        if cand in allowed:
            return cand
    for y, m in RE_KOR_LINE.findall(text):
        cand = f"{y}-{int(m):02d}"
        if cand in allowed:
            return cand
    m = re.search(r'(20\d{2})-(0[1-9]|1[0-2])', text)
    if m:
        cand = f"{m.group(1)}-{m.group(2)}"
        if cand in allowed:
            return cand
    return ""

# -----------------------------------------
# 4) 페르소나 TXT 파싱
# -----------------------------------------
BLOCK_RE = re.compile(
    r"\*\*Persona_(\d{1,3})\*\*\s*(.+?)(?=(?:\n\s*\*\*Persona_\d{1,3}\*\*|\Z))",
    re.DOTALL
)

def parse_personas(text: str) -> List[Tuple[str, str]]:
    personas: List[Tuple[str, str]] = []
    for pid, body in BLOCK_RE.findall(text):
        personas.append((f"Persona_{int(pid):03d}", body.strip()))
    return personas

# -----------------------------------------
# 5) 단일 페르소나 질의
# -----------------------------------------
def ask_month_for_persona(
    pipe,
    persona_text: str,
    temperature: float = 1.20,
    top_p: float = 0.97,
    top_k: int = 0,
    max_new_tokens: int = 12,
    verbose: bool = False
) -> Tuple[str, str]:
    prompt = build_single_month_prompt(persona_text, ALLOWED_MONTHS)
    out = pipe(
        [{"role": "user", "content": prompt}],
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=temperature,
        top_p=top_p,
        top_k=top_k,
        repetition_penalty=1.0,
        return_full_text=False
    )
    raw = out[0].get("generated_text", out[0].get("text", "")).strip()
    month = parse_one_month(raw, ALLOWED_MONTHS) or ""
    if verbose:
        print(f"[RAW] {raw} -> [PARSED] {month or '(fail)'}")
    return month, raw

# -----------------------------------------
# 6) 전체 실행 루프
# -----------------------------------------
def run_survey_over_personas(
    pipe,
    personas_text: str,
    save_dir: str = ".",
    show_progress: bool = True,
    print_details: bool = True
) -> Dict[str, int]:
    os.makedirs(save_dir, exist_ok=True)

    personas = parse_personas(personas_text)
    if not personas:
        raise ValueError("페르소나를 찾지 못했으니 입력 형식을 확인하라.")

    month_counts: Dict[str, int] = {m: 0 for m in ALLOWED_MONTHS}
    total = len(personas)

    log_path = os.path.join(save_dir, "persona_choices.csv")
    with open(log_path, "w", newline="", encoding="utf-8") as f:
        wr = csv.writer(f)
        wr.writerow(["persona_id", "chosen_month", "raw_output"])

        for i, (pid, body) in enumerate(personas, start=1):
            if show_progress:
                print(f"[{i}/{total}] {pid} 질문 중...", flush=True)

            try:
                chosen, raw = ask_month_for_persona(pipe, body, verbose=False)
            except Exception as e:
                chosen, raw = "", f"__ERROR__: {e!r}"

            if chosen:
                month_counts[chosen] += 1
            wr.writerow([pid, chosen, raw])

            if show_progress and print_details:
                short_raw = (raw[:80] + "…") if len(raw) > 80 else raw
                print(f"   ↳ 선택: {chosen or '(파싱 실패)'} | 원문: {short_raw}", flush=True)

    cnt_path = os.path.join(save_dir, "month_counts_Spicy_Oil_135g.csv")
    with open(cnt_path, "w", newline="", encoding="utf-8") as f:
        wr = csv.writer(f)
        wr.writerow(["month", "count"])
        for m in ALLOWED_MONTHS:
            wr.writerow([m, month_counts[m]])

    print("\n[월별 선택 횟수]")
    for m in ALLOWED_MONTHS:
        print(f"{m}: {month_counts[m]}")

    print(f"\n저장 완료:\n- {log_path}\n- {cnt_path}")
    return month_counts

# -----------------------------------------
# 7) 실행
# -----------------------------------------
if __name__ == "__main__":
    month_counts = run_survey_over_personas(
        pipe,
        PERSONAS_RAW,
        save_dir=".",
        show_progress=True,
        print_details=True
    )


In [ ]:
import pandas as pd

# CSV 파일 불러오기
df = pd.read_csv("month_counts_Spicy_Oil_135g.csv", encoding="utf-8")

# 평균 count 계산
mean_count = df["count"].mean()

# factor 및 보정된 adj_factor 계산
df["factor"] = df["count"] / mean_count
df["adj_factor"] = 1 + (df["factor"] -1) / 100

print(df)

# 월별 가중치를 위한 Simulation : 동원맛참 매콤참기름 90g

In [ ]:
# -*- coding: utf-8 -*-
import re, csv, os
from typing import List, Tuple, Dict

# -----------------------------------------
# 0) 후보 달 (2024-04 ~ 2025-06)
# -----------------------------------------
ALLOWED_MONTHS: List[str] = [
    "2024-04","2024-05","2024-06","2024-07","2024-08","2024-09",
    "2024-10","2024-11","2024-12","2025-01","2025-02","2025-03",
    "2025-04","2025-05","2025-06"
]

# -----------------------------------------
# 1) 속성 중요도
# -----------------------------------------
IMPORTANCE_ORDER = [
    "요리 여부·빈도",             # 속성8
    "월평균 가구 소득 수준",       # 속성5
    "참치캔 주 사용 용도",         # 속성7
    "가구원수",                   # 속성3
    "자녀 유무",                   # 속성4
    "선호하는 참치캔 브랜드",      # 속성10
    "구매 채널",                   # 속성6
    "성별",                        # 속성2
    "선호하는 참치캔 타입",        # 속성9
    "나이"                         # 속성1
]

# -----------------------------------------
# 1-1) 대상 제품/설명
# -----------------------------------------
PRODUCT_NAME = "동원맛참 매콤참기름 90g"
PRODUCT_DESC = (
    "‘동원맛참 매콤참기름’은 고소한 참깨 본연의 풍미에 청양고추와 고추기름을 더해 매콤한 맛을 구현한 프리미엄 참기름 제품이다. "
    "단백질과 필수 미네랄인 셀레늄을 함유해 맛뿐 아니라 건강까지 고려할 수 있으며, "
    "기존 참기름의 고소함과 함께 매콤한 풍미가 어우러져 비빔밥·국수·볶음·무침 등 다양한 한식 요리에 색다른 매력을 더한다. "
    "캡사이신 대신 실제 고추를 우려내어 자극적이지 않고 깔끔하게 오래 남는 매운맛이 특징이다. "
    "2025년 상반기에는 ‘한 방울로 매콤한 변화’를 콘셉트로 TV, YouTube, SNS를 아우르는 마케팅 캠페인을 전개했으며, "
    "수도권 대형마트에서는 시식 행사와 쿠킹 클래스 협업으로 소비자 경험을 강화하였다. "
    "광고 모델로는 그룹 아이브(IVE)의 안유진을 기용해 MZ세대에게 세련되고 건강한 이미지를 각인시켰으며, "
    "참기름의 고소함과 매콤함을 결합한 새로운 카테고리 제품으로 자리매김하고 있다."
)

# -----------------------------------------
# 2) 프롬프트 (대상 제품 한정)
# -----------------------------------------
def build_single_month_prompt(persona_text: str, months: List[str]) -> str:
    months_line = ", ".join(months)
    importance_line = " > ".join(IMPORTANCE_ORDER)
    return f"""
너는 한국 소비자 그릭요거트 구매·섭취 행동을 분석하는 시장조사 전문가다.

[제품명]
{PRODUCT_NAME}

[제품 설명]
{PRODUCT_DESC}

[페르소나]
{persona_text.strip()}

[질문]
아래 달 중에서 이 페르소나가 위의 제품({PRODUCT_NAME})을 가장 많이 살 것 같은 달 1개만 고르라.
판단 시 아래 속성 중요도(상→하)를 우선 고려하라: {importance_line}

[선택 가능한 달(반드시 이 중에서만 선택)]
{months_line}

[출력 형식(다른 말 절대 금지)]
YYYY-MM
""".strip()

# -----------------------------------------
# 3) 응답 파서
# -----------------------------------------
RE_ISO_LINE  = re.compile(r'^\s*(20\d{2})-(0[1-9]|1[0-2])\s*$', re.MULTILINE)
RE_KOR_LINE  = re.compile(r'^\s*(20\d{2})\s*년\s*(1[0-2]|[1-9])\s*월\s*$', re.MULTILINE)

def parse_one_month(text: str, allowed: List[str]) -> str:
    for y, m in RE_ISO_LINE.findall(text):
        cand = f"{y}-{m}"
        if cand in allowed:
            return cand
    for y, m in RE_KOR_LINE.findall(text):
        cand = f"{y}-{int(m):02d}"
        if cand in allowed:
            return cand
    m = re.search(r'(20\d{2})-(0[1-9]|1[0-2])', text)
    if m:
        cand = f"{m.group(1)}-{m.group(2)}"
        if cand in allowed:
            return cand
    return ""

# -----------------------------------------
# 4) 페르소나 TXT 파싱
# -----------------------------------------
BLOCK_RE = re.compile(
    r"\*\*Persona_(\d{1,3})\*\*\s*(.+?)(?=(?:\n\s*\*\*Persona_\d{1,3}\*\*|\Z))",
    re.DOTALL
)

def parse_personas(text: str) -> List[Tuple[str, str]]:
    personas: List[Tuple[str, str]] = []
    for pid, body in BLOCK_RE.findall(text):
        personas.append((f"Persona_{int(pid):03d}", body.strip()))
    return personas

# -----------------------------------------
# 5) 단일 페르소나 질의
# -----------------------------------------
def ask_month_for_persona(
    pipe,
    persona_text: str,
    temperature: float = 1.20,
    top_p: float = 0.97,
    top_k: int = 0,
    max_new_tokens: int = 12,
    verbose: bool = False
) -> Tuple[str, str]:
    prompt = build_single_month_prompt(persona_text, ALLOWED_MONTHS)
    out = pipe(
        [{"role": "user", "content": prompt}],
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=temperature,
        top_p=top_p,
        top_k=top_k,
        repetition_penalty=1.0,
        return_full_text=False
    )
    raw = out[0].get("generated_text", out[0].get("text", "")).strip()
    month = parse_one_month(raw, ALLOWED_MONTHS) or ""
    if verbose:
        print(f"[RAW] {raw} -> [PARSED] {month or '(fail)'}")
    return month, raw

# -----------------------------------------
# 6) 전체 실행 루프
# -----------------------------------------
def run_survey_over_personas(
    pipe,
    personas_text: str,
    save_dir: str = ".",
    show_progress: bool = True,
    print_details: bool = True
) -> Dict[str, int]:
    os.makedirs(save_dir, exist_ok=True)

    personas = parse_personas(personas_text)
    if not personas:
        raise ValueError("페르소나를 찾지 못했으니 입력 형식을 확인하라.")

    month_counts: Dict[str, int] = {m: 0 for m in ALLOWED_MONTHS}
    total = len(personas)

    log_path = os.path.join(save_dir, "persona_choices.csv")
    with open(log_path, "w", newline="", encoding="utf-8") as f:
        wr = csv.writer(f)
        wr.writerow(["persona_id", "chosen_month", "raw_output"])

        for i, (pid, body) in enumerate(personas, start=1):
            if show_progress:
                print(f"[{i}/{total}] {pid} 질문 중...", flush=True)

            try:
                chosen, raw = ask_month_for_persona(pipe, body, verbose=False)
            except Exception as e:
                chosen, raw = "", f"__ERROR__: {e!r}"

            if chosen:
                month_counts[chosen] += 1
            wr.writerow([pid, chosen, raw])

            if show_progress and print_details:
                short_raw = (raw[:80] + "…") if len(raw) > 80 else raw
                print(f"   ↳ 선택: {chosen or '(파싱 실패)'} | 원문: {short_raw}", flush=True)

    cnt_path = os.path.join(save_dir, "month_counts_Spicy_Oil_90g.csv")
    with open(cnt_path, "w", newline="", encoding="utf-8") as f:
        wr = csv.writer(f)
        wr.writerow(["month", "count"])
        for m in ALLOWED_MONTHS:
            wr.writerow([m, month_counts[m]])

    print("\n[월별 선택 횟수]")
    for m in ALLOWED_MONTHS:
        print(f"{m}: {month_counts[m]}")

    print(f"\n저장 완료:\n- {log_path}\n- {cnt_path}")
    return month_counts

# -----------------------------------------
# 7) 실행
# -----------------------------------------
if __name__ == "__main__":
    month_counts = run_survey_over_personas(
        pipe,
        PERSONAS_RAW,
        save_dir=".",
        show_progress=True,
        print_details=True
    )


In [ ]:
import pandas as pd

# CSV 파일 불러오기
df = pd.read_csv("month_counts_Spicy_Oil_90g.csv", encoding="utf-8")

# 평균 count 계산
mean_count = df["count"].mean()

# factor 및 보정된 adj_factor 계산
df["factor"] = df["count"] / mean_count
df["adj_factor"] = 1 + (df["factor"] -1) / 100

print(df)

# **그릭 요거트**

> # 생성한 요거트 페르소나 불러오기

In [ ]:
import re, csv, os
from typing import List, Tuple

with open("GreekYogurt.txt", "r", encoding="utf-8") as f:
    PERSONAS_RAW = f.read()

print("\n".join(PERSONAS_RAW.splitlines()[:5]))

# 속성별 가중치 계산 : 순위로

In [ ]:
# -*- coding: utf-8 -*-
import re
from typing import List, Dict, Tuple

# -----------------------------------------
# 0) 속성 정의 (그릭요거트 페르소나용)
# -----------------------------------------
ATTRS: Dict[int, Tuple[str, str]] = {
    1: ("나이", "10대 / 20대 / 30대 / 40-50대 / 60대 이상"),
    2: ("성별", "남성 / 여성"),
    3: ("가구원수", "혼자 거주하는 1인 가구 / 부부가 함께 사는 2인 가구 / 부모와 자녀 세대가 함께 사는 2세대 이상 가구"),
    4: ("자녀 유무", "있음 / 없음"),
    5: ("월평균 가구 소득 수준", "월평균 가구 소득이 200만원 이상 350만원 미만인 저소득층 / 월평균 가구 소득이 350만원 이상 700만원 이하인 중간 소득층 / 월평균 가구 소득이 700만원을 초과하는 고소득층"),
    6: ("구매 채널", "대형마트와 같은 오프라인 / 쿠팡·마켓컬리·네이버와 같은 온라인 몰 / 온라인+오프라인"),
    7: ("최근 1주일 외식·배달 횟수", "0회 / 1-2회 / 3-4회 / 5회 이상"),
    8: ("섭취 빈도", "거의 안 먹음(월 1회 이하) / 가끔(주 1-2회) / 자주(주 3회 이상)"),
    9: ("그릭요거트 주 사용 용도", "샐러드·토핑 / 간식·디저트 / 다이어트·단백질 보충"),
    10: ("선호하는 그릭요거트 브랜드", "덴마크(동원) / 매일 / 피코크 / 풀무원 / 기타 브랜드"),
}

# -----------------------------------------
# 옵션: 라운드별 로그 출력 여부 (기본 False)
# -----------------------------------------
VERBOSE = False

# -----------------------------------------
# 1) 프롬프트 생성기
#    - 모델 출력: 반드시 "속성n" (예: 속성10)
# -----------------------------------------
def build_single_pick_prompt(remaining_ids: List[int]) -> str:
    lines = []
    for i in remaining_ids:
        short, detail = ATTRS[i]
        lines.append(f"{i}. {short} — {detail}")
    attr_block = "\n".join(lines)

    prompt = f"""
너는 그릭요거트 섭취·구매 행동을 분석하는 시장조사 전문가다.
아래 속성 후보들 중에서 '그릭요거트 섭취·구매'에 가장 큰 영향을 미치는 속성 하나만 고르라.

[후보 속성]
{attr_block}

[출력 형식(다른 말 절대 금지)]
속성[번호]

예시:
속성10
""".strip()
    return prompt

# -----------------------------------------
# 2) 파서: "속성n" 한 줄만 허용
#    - (안전장치로 숫자만 응답해도 인식)
# -----------------------------------------
PAT_PROP = re.compile(r'^\s*속성\s*(\d{1,2})\s*$', re.IGNORECASE)
PAT_NUM  = re.compile(r'^\s*(\d{1,2})\s*$')

def parse_single_pick(text: str, allowed: List[int]) -> int:
    txt = text.strip()
    m = PAT_PROP.match(txt)
    if m:
        n = int(m.group(1))
        return n if n in allowed else -1
    m2 = PAT_NUM.match(txt)  # 예외적으로 숫자만 줄 경우
    if m2:
        n = int(m2.group(1))
        return n if n in allowed else -1
    return -1

# -----------------------------------------
# 3) 제거식 랭킹
# -----------------------------------------
def get_greek_yogurt_importance_elimination(pipe) -> List[int]:
    remaining = list(range(1, 11))
    ranking = []

    round_idx = 1
    while remaining:
        prompt = build_single_pick_prompt(remaining)
        out = pipe(
            [{"role": "user", "content": prompt}],
            max_new_tokens=8,          # "속성n"만 받으면 충분
            do_sample=False,           # 결정적 응답
            return_full_text=False
        )
        raw = out[0].get("generated_text", out[0].get("text", "")).strip()
        picked = parse_single_pick(raw, remaining)

        # 파싱 실패 시 보수적 폴백
        if picked == -1:
            picked = remaining[0]
            if VERBOSE:
                print(f"[라운드 {round_idx}] 파싱 실패 → 폴백 사용, 선택: 속성{picked}")

        if VERBOSE:
            print(f"[라운드 {round_idx}] 모델 출력: {raw}  → 선택: 속성{picked}")

        ranking.append(picked)
        remaining.remove(picked)
        round_idx += 1

    # 최종 요약만 출력
    print("\n[속성별 중요도 순위]  (1위 → 10위)")
    for pos, idx in enumerate(ranking, start=1):
        short, detail = ATTRS[idx]
        print(f"{pos}위: 속성{idx} — {short}")
    return ranking

# -----------------------------------------
# 4) 실행부 (이미 pipe 로드됨)
# -----------------------------------------
if __name__ == "__main__":
    ranking_elim = get_greek_yogurt_importance_elimination(pipe)


# 월별 가중치를 위한 Simulation : 덴마크 하이그릭요거트 400g

In [ ]:
# -*- coding: utf-8 -*-
import re, csv, os
from typing import List, Tuple, Dict

# -----------------------------------------
# 0) 후보 달 (2024-04 ~ 2025-06)
# -----------------------------------------
ALLOWED_MONTHS: List[str] = [
    "2024-04","2024-05","2024-06","2024-07","2024-08","2024-09",
    "2024-10","2024-11","2024-12","2025-01","2025-02","2025-03",
    "2025-04","2025-05","2025-06"
]

# -----------------------------------------
# 1) 속성 중요도
# -----------------------------------------
IMPORTANCE_ORDER = [
    "섭취 빈도",                   # 속성8
    "월평균 가구 소득 수준",       # 속성5
    "그릭요거트 주 사용 용도",     # 속성9
    "최근 1주일 외식·배달 횟수",   # 속성7
    "가구원수",                   # 속성3
    "구매 채널",                   # 속성6
    "선호하는 그릭요거트 브랜드",  # 속성10
    "자녀 유무",                   # 속성4
    "나이",                        # 속성1
    "성별"                         # 속성2
]

# -----------------------------------------
# 1-1) 대상 제품/설명
# -----------------------------------------
PRODUCT_NAME = "덴마크 하이그릭요거트 400g"
PRODUCT_DESC = (
    "‘덴마크 하이그릭요거트 400g’은 이중 유청분리 공법을 적용해 수분을 줄이고, "
    "꾸덕꾸덕하고 진한 질감을 살린 프리미엄 호상(떠먹는) 발효유 제품이다. "
    "단백질이 풍부해 고단백 식품으로 자리매김하고 있으며, 아연과 칼슘 등 필수 미네랄을 함유해 "
    "건강과 영양을 동시에 챙길 수 있다. "
    "고소하고 진한 풍미가 특징으로, 아침 식사 대용이나 샐러드·토핑, 다이어트 간식으로 적합하다. "
    "2025년 6~7월에는 TV, YouTube, SNS를 통한 대대적인 광고 캠페인을 진행했으며, "
    "6~8월에는 수도권 아파트 단지 엘리베이터 광고를 통해 소비자 접점을 강화하였다. "
    "광고 모델로는 전문 연예인 대신 일반인을 기용해 친근감과 현실적인 소비자 공감을 이끌어낸 점도 특징이다."
)


# -----------------------------------------
# 2) 프롬프트 (대상 제품 한정)
# -----------------------------------------
def build_single_month_prompt(persona_text: str, months: List[str]) -> str:
    months_line = ", ".join(months)
    importance_line = " > ".join(IMPORTANCE_ORDER)
    return f"""
너는 한국 소비자 그릭요거트 구매·섭취 행동을 분석하는 시장조사 전문가다.

[제품명]
{PRODUCT_NAME}

[제품 설명]
{PRODUCT_DESC}

[페르소나]
{persona_text.strip()}

[질문]
아래 달 중에서 이 페르소나가 위의 제품({PRODUCT_NAME})을 가장 많이 살 것 같은 달 1개만 고르라.
판단 시 아래 속성 중요도(상→하)를 우선 고려하라: {importance_line}

[선택 가능한 달(반드시 이 중에서만 선택)]
{months_line}

[출력 형식(다른 말 절대 금지)]
YYYY-MM
""".strip()

# -----------------------------------------
# 3) 응답 파서
# -----------------------------------------
RE_ISO_LINE  = re.compile(r'^\s*(20\d{2})-(0[1-9]|1[0-2])\s*$', re.MULTILINE)
RE_KOR_LINE  = re.compile(r'^\s*(20\d{2})\s*년\s*(1[0-2]|[1-9])\s*월\s*$', re.MULTILINE)

def parse_one_month(text: str, allowed: List[str]) -> str:
    for y, m in RE_ISO_LINE.findall(text):
        cand = f"{y}-{m}"
        if cand in allowed:
            return cand
    for y, m in RE_KOR_LINE.findall(text):
        cand = f"{y}-{int(m):02d}"
        if cand in allowed:
            return cand
    m = re.search(r'(20\d{2})-(0[1-9]|1[0-2])', text)
    if m:
        cand = f"{m.group(1)}-{m.group(2)}"
        if cand in allowed:
            return cand
    return ""

# -----------------------------------------
# 4) 페르소나 TXT 파싱
# -----------------------------------------
BLOCK_RE = re.compile(
    r"\*\*Persona_(\d{1,3})\*\*\s*(.+?)(?=(?:\n\s*\*\*Persona_\d{1,3}\*\*|\Z))",
    re.DOTALL
)

def parse_personas(text: str) -> List[Tuple[str, str]]:
    personas: List[Tuple[str, str]] = []
    for pid, body in BLOCK_RE.findall(text):
        personas.append((f"Persona_{int(pid):03d}", body.strip()))
    return personas

# -----------------------------------------
# 5) 단일 페르소나 질의
# -----------------------------------------
def ask_month_for_persona(
    pipe,
    persona_text: str,
    temperature: float = 1.20,
    top_p: float = 0.97,
    top_k: int = 0,
    max_new_tokens: int = 12,
    verbose: bool = False
) -> Tuple[str, str]:
    prompt = build_single_month_prompt(persona_text, ALLOWED_MONTHS)
    out = pipe(
        [{"role": "user", "content": prompt}],
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=temperature,
        top_p=top_p,
        top_k=top_k,
        repetition_penalty=1.0,
        return_full_text=False
    )
    raw = out[0].get("generated_text", out[0].get("text", "")).strip()
    month = parse_one_month(raw, ALLOWED_MONTHS) or ""
    if verbose:
        print(f"[RAW] {raw} -> [PARSED] {month or '(fail)'}")
    return month, raw

# -----------------------------------------
# 6) 전체 실행 루프
# -----------------------------------------
def run_survey_over_personas(
    pipe,
    personas_text: str,
    save_dir: str = ".",
    show_progress: bool = True,
    print_details: bool = True
) -> Dict[str, int]:
    os.makedirs(save_dir, exist_ok=True)

    personas = parse_personas(personas_text)
    if not personas:
        raise ValueError("페르소나를 찾지 못했으니 입력 형식을 확인하라.")

    month_counts: Dict[str, int] = {m: 0 for m in ALLOWED_MONTHS}
    total = len(personas)

    log_path = os.path.join(save_dir, "persona_choices.csv")
    with open(log_path, "w", newline="", encoding="utf-8") as f:
        wr = csv.writer(f)
        wr.writerow(["persona_id", "chosen_month", "raw_output"])

        for i, (pid, body) in enumerate(personas, start=1):
            if show_progress:
                print(f"[{i}/{total}] {pid} 질문 중...", flush=True)

            try:
                chosen, raw = ask_month_for_persona(pipe, body, verbose=False)
            except Exception as e:
                chosen, raw = "", f"__ERROR__: {e!r}"

            if chosen:
                month_counts[chosen] += 1
            wr.writerow([pid, chosen, raw])

            if show_progress and print_details:
                short_raw = (raw[:80] + "…") if len(raw) > 80 else raw
                print(f"   ↳ 선택: {chosen or '(파싱 실패)'} | 원문: {short_raw}", flush=True)

    cnt_path = os.path.join(save_dir, "month_counts_GreekYogurt.csv")
    with open(cnt_path, "w", newline="", encoding="utf-8") as f:
        wr = csv.writer(f)
        wr.writerow(["month", "count"])
        for m in ALLOWED_MONTHS:
            wr.writerow([m, month_counts[m]])

    print("\n[월별 선택 횟수]")
    for m in ALLOWED_MONTHS:
        print(f"{m}: {month_counts[m]}")

    print(f"\n저장 완료:\n- {log_path}\n- {cnt_path}")
    return month_counts

# -----------------------------------------
# 7) 실행
# -----------------------------------------
if __name__ == "__main__":
    month_counts = run_survey_over_personas(
        pipe,
        PERSONAS_RAW,
        save_dir=".",
        show_progress=True,
        print_details=True
    )


In [ ]:
import pandas as pd

# CSV 파일 불러오기
df = pd.read_csv("month_counts_GreekYogurt.csv", encoding="utf-8")

# 평균 count 계산
mean_count = df["count"].mean()

# factor 및 보정된 adj_factor 계산
df["factor"] = df["count"] / mean_count
df["adj_factor"] = 1 + (df["factor"] -1) / 100

print(df)

# **오믈레햄**

> # 생성한 오믈레햄 페르소나 불러오기

In [ ]:
import re, csv, os
from typing import List, Tuple

with open("OmletHam.txt", "r", encoding="utf-8") as f:
    PERSONAS_RAW = f.read()

print("\n".join(PERSONAS_RAW.splitlines()[:5]))

# 속성별 가중치 계산 : 순위로

In [ ]:
# -*- coding: utf-8 -*-
import re
from typing import List, Dict, Tuple

# -----------------------------------------
# 0) 속성 정의 (신제품 햄 페르소나용)
# -----------------------------------------
ATTRS: Dict[int, Tuple[str, str]] = {
    1: ("나이", "10대 / 20대 / 30대 / 40-50대 / 60대 이상"),
    2: ("성별", "남성 / 여성"),
    3: ("가구원수", "혼자 거주하는 1인 가구 / 부부가 함께 사는 2인 가구 / 부모와 자녀 세대가 함께 사는 2세대 이상 가구"),
    4: ("자녀 유무", "있음 / 없음"),
    5: ("월평균 가구 소득 수준", "월평균 가구 소득이 200만원 이상 350만원 미만인 저소득층 / 월평균 가구 소득이 350만원 이상 700만원 이하인 중간 소득층 / 월평균 가구 소득이 700만원을 초과하는 고소득층"),
    6: ("구매 채널", "대형마트와 같은 오프라인 / 쿠팡·마켓컬리·네이버와 같은 온라인 몰 / 온라인+오프라인"),
    7: ("선호하는 통조림 햄 타입", "클래식 / 저염(라이트) / 기타(치즈·갈릭·오믈레 등)"),
    8: ("요리 여부·빈도", "전혀 하지 않음 / 가끔(주 1-2회) / 자주(주 3회 이상)"),
    9: ("통조림 햄 주 사용 용도", "찌개 / 샌드위치·토스트 / 반찬·볶음밥"),
    10: ("선호하는 통조림 햄 브랜드", "동원 / CJ / 롯데 / 기타 브랜드"),
}

# -----------------------------------------
# 옵션: 라운드별 로그 출력 여부 (기본 False)
# -----------------------------------------
VERBOSE = False

# -----------------------------------------
# 1) 프롬프트 생성기
#    - 모델 출력: 반드시 "속성n" (예: 속성10)
# -----------------------------------------
def build_single_pick_prompt(remaining_ids: List[int]) -> str:
    lines = []
    for i in remaining_ids:
        short, detail = ATTRS[i]
        lines.append(f"{i}. {short} — {detail}")
    attr_block = "\n".join(lines)

    prompt = f"""
너는 신제품 햄(통조림 햄) 섭취·구매 행동을 분석하는 시장조사 전문가다.
아래 속성 후보들 중에서 '신제품 햄(통조림 햄) 섭취·구매'에 가장 큰 영향을 미치는 속성 하나만 고르라.

[후보 속성]
{attr_block}

[출력 형식(다른 말 절대 금지)]
속성[번호]

예시:
속성10
""".strip()
    return prompt

# -----------------------------------------
# 2) 파서: "속성n" 한 줄만 허용
# -----------------------------------------
PAT_PROP = re.compile(r'^\s*속성\s*(\d{1,2})\s*$', re.IGNORECASE)
PAT_NUM  = re.compile(r'^\s*(\d{1,2})\s*$')

def parse_single_pick(text: str, allowed: List[int]) -> int:
    txt = text.strip()
    m = PAT_PROP.match(txt)
    if m:
        n = int(m.group(1))
        return n if n in allowed else -1
    m2 = PAT_NUM.match(txt)  # 숫자만 줄 경우 허용
    if m2:
        n = int(m2.group(1))
        return n if n in allowed else -1
    return -1

# -----------------------------------------
# 3) 제거식 랭킹
# -----------------------------------------
def get_ham_importance_elimination(pipe) -> List[int]:
    remaining = list(range(1, 11))
    ranking = []

    round_idx = 1
    while remaining:
        prompt = build_single_pick_prompt(remaining)
        out = pipe(
            [{"role": "user", "content": prompt}],
            max_new_tokens=8,
            do_sample=False,
            return_full_text=False
        )
        raw = out[0].get("generated_text", out[0].get("text", "")).strip()
        picked = parse_single_pick(raw, remaining)

        if picked == -1:
            picked = remaining[0]  # 보수적 폴백
            if VERBOSE:
                print(f"[라운드 {round_idx}] 파싱 실패 → 폴백 사용, 선택: 속성{picked}")

        if VERBOSE:
            print(f"[라운드 {round_idx}] 모델 출력: {raw}  → 선택: 속성{picked}")

        ranking.append(picked)
        remaining.remove(picked)
        round_idx += 1

    # 최종 요약만 출력
    print("\n[속성별 중요도 순위]  (1위 → 10위)")
    for pos, idx in enumerate(ranking, start=1):
        short, detail = ATTRS[idx]
        print(f"{pos}위: 속성{idx} — {short}")
    return ranking

# -----------------------------------------
# 4) 실행부 (이미 pipe 로드됨)
# -----------------------------------------
if __name__ == "__main__":
    ranking_elim = get_ham_importance_elimination(pipe)


# 월별 가중치를 위한 Simulation : 리챔 오믈레햄 200g

In [ ]:
# -*- coding: utf-8 -*-
import re, csv, os
from typing import List, Tuple, Dict

# -----------------------------------------
# 0) 후보 달 (2024-04 ~ 2025-06)
# -----------------------------------------
ALLOWED_MONTHS: List[str] = [
    "2024-04","2024-05","2024-06","2024-07","2024-08","2024-09",
    "2024-10","2024-11","2024-12","2025-01","2025-02","2025-03",
    "2025-04","2025-05","2025-06"
]

# -----------------------------------------
# 1) 속성 중요도
# -----------------------------------------
IMPORTANCE_ORDER = [
    "요리 여부·빈도",              # 속성8
    "월평균 가구 소득 수준",       # 속성5
    "가구원수",                   # 속성3
    "선호하는 통조림 햄 타입",     # 속성7
    "통조림 햄 주 사용 용도",      # 속성9
    "선호하는 통조림 햄 브랜드",   # 속성10
    "자녀 유무",                   # 속성4
    "구매 채널",                   # 속성6
    "나이",                        # 속성1
    "성별"                         # 속성2
]


# -----------------------------------------
# 1-1) 대상 제품/설명
# -----------------------------------------
PRODUCT_NAME = "리챔 오믈레햄 200g"
PRODUCT_DESC = (
    "‘리챔 오믈레햄 200g’은 오믈렛(Omelet)과 햄(Ham)의 합성 개념으로 개발된 신개념 통조림 햄 제품이다. "
    "저나트륨 설계를 통해 기존 햄 대비 부담을 줄였으며, 내열성 케첩 소스를 적용해 조리 시 풍미가 살아나는 것이 특징이다. "
    "한 끼 반찬이나 간편 도시락, 캠핑·야외활동 등 다양한 상황에서 활용하기 적합하며, "
    "부드러운 식감과 은은한 토마토 풍미가 더해져 아이들 간식부터 어른들의 반주 안주까지 폭넓게 어울린다. "
    "2025년 상반기에는 온라인몰 시식 리뷰 이벤트와 편의점 시식 프로모션을 통해 입소문을 확산시켰으며, "
    "소비자들에게 ‘간편하지만 특별한 한 끼’라는 이미지를 각인시키는 전략을 펼쳤다."
)

# -----------------------------------------
# 2) 프롬프트 (대상 제품 한정)
# -----------------------------------------
def build_single_month_prompt(persona_text: str, months: List[str]) -> str:
    months_line = ", ".join(months)
    importance_line = " > ".join(IMPORTANCE_ORDER)
    return f"""
너는 한국 소비자 신제품 햄 구매·섭취 행동을 분석하는 시장조사 전문가다.

[제품명]
{PRODUCT_NAME}

[제품 설명]
{PRODUCT_DESC}

[페르소나]
{persona_text.strip()}

[질문]
아래 달 중에서 이 페르소나가 위의 제품({PRODUCT_NAME})을 가장 많이 살 것 같은 달 1개만 고르라.
판단 시 아래 속성 중요도(상→하)를 우선 고려하라: {importance_line}

[선택 가능한 달(반드시 이 중에서만 선택)]
{months_line}

[출력 형식(다른 말 절대 금지)]
YYYY-MM
""".strip()

# -----------------------------------------
# 3) 응답 파서
# -----------------------------------------
RE_ISO_LINE  = re.compile(r'^\s*(20\d{2})-(0[1-9]|1[0-2])\s*$', re.MULTILINE)
RE_KOR_LINE  = re.compile(r'^\s*(20\d{2})\s*년\s*(1[0-2]|[1-9])\s*월\s*$', re.MULTILINE)

def parse_one_month(text: str, allowed: List[str]) -> str:
    for y, m in RE_ISO_LINE.findall(text):
        cand = f"{y}-{m}"
        if cand in allowed:
            return cand
    for y, m in RE_KOR_LINE.findall(text):
        cand = f"{y}-{int(m):02d}"
        if cand in allowed:
            return cand
    m = re.search(r'(20\d{2})-(0[1-9]|1[0-2])', text)
    if m:
        cand = f"{m.group(1)}-{m.group(2)}"
        if cand in allowed:
            return cand
    return ""

# -----------------------------------------
# 4) 페르소나 TXT 파싱
# -----------------------------------------
BLOCK_RE = re.compile(
    r"\*\*Persona_(\d{1,3})\*\*\s*(.+?)(?=(?:\n\s*\*\*Persona_\d{1,3}\*\*|\Z))",
    re.DOTALL
)

def parse_personas(text: str) -> List[Tuple[str, str]]:
    personas: List[Tuple[str, str]] = []
    for pid, body in BLOCK_RE.findall(text):
        personas.append((f"Persona_{int(pid):03d}", body.strip()))
    return personas

# -----------------------------------------
# 5) 단일 페르소나 질의
# -----------------------------------------
def ask_month_for_persona(
    pipe,
    persona_text: str,
    temperature: float = 1.20,
    top_p: float = 0.97,
    top_k: int = 0,
    max_new_tokens: int = 12,
    verbose: bool = False
) -> Tuple[str, str]:
    prompt = build_single_month_prompt(persona_text, ALLOWED_MONTHS)
    out = pipe(
        [{"role": "user", "content": prompt}],
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=temperature,
        top_p=top_p,
        top_k=top_k,
        repetition_penalty=1.0,
        return_full_text=False
    )
    raw = out[0].get("generated_text", out[0].get("text", "")).strip()
    month = parse_one_month(raw, ALLOWED_MONTHS) or ""
    if verbose:
        print(f"[RAW] {raw} -> [PARSED] {month or '(fail)'}")
    return month, raw

# -----------------------------------------
# 6) 전체 실행 루프
# -----------------------------------------
def run_survey_over_personas(
    pipe,
    personas_text: str,
    save_dir: str = ".",
    show_progress: bool = True,
    print_details: bool = True
) -> Dict[str, int]:
    os.makedirs(save_dir, exist_ok=True)

    personas = parse_personas(personas_text)
    if not personas:
        raise ValueError("페르소나를 찾지 못했으니 입력 형식을 확인하라.")

    month_counts: Dict[str, int] = {m: 0 for m in ALLOWED_MONTHS}
    total = len(personas)

    log_path = os.path.join(save_dir, "persona_choices.csv")
    with open(log_path, "w", newline="", encoding="utf-8") as f:
        wr = csv.writer(f)
        wr.writerow(["persona_id", "chosen_month", "raw_output"])

        for i, (pid, body) in enumerate(personas, start=1):
            if show_progress:
                print(f"[{i}/{total}] {pid} 질문 중...", flush=True)

            try:
                chosen, raw = ask_month_for_persona(pipe, body, verbose=False)
            except Exception as e:
                chosen, raw = "", f"__ERROR__: {e!r}"

            if chosen:
                month_counts[chosen] += 1
            wr.writerow([pid, chosen, raw])

            if show_progress and print_details:
                short_raw = (raw[:80] + "…") if len(raw) > 80 else raw
                print(f"   ↳ 선택: {chosen or '(파싱 실패)'} | 원문: {short_raw}", flush=True)

    cnt_path = os.path.join(save_dir, "month_counts_Omlet_200g.csv")
    with open(cnt_path, "w", newline="", encoding="utf-8") as f:
        wr = csv.writer(f)
        wr.writerow(["month", "count"])
        for m in ALLOWED_MONTHS:
            wr.writerow([m, month_counts[m]])

    print("\n[월별 선택 횟수]")
    for m in ALLOWED_MONTHS:
        print(f"{m}: {month_counts[m]}")

    print(f"\n저장 완료:\n- {log_path}\n- {cnt_path}")
    return month_counts

# -----------------------------------------
# 7) 실행
# -----------------------------------------
if __name__ == "__main__":
    month_counts = run_survey_over_personas(
        pipe,
        PERSONAS_RAW,
        save_dir=".",
        show_progress=True,
        print_details=True
    )


In [ ]:
import pandas as pd

# CSV 파일 불러오기
df = pd.read_csv("month_counts_Omlet_200g.csv", encoding="utf-8")

# 평균 count 계산
mean_count = df["count"].mean()

# factor 및 보정된 adj_factor 계산
df["factor"] = df["count"] / mean_count
df["adj_factor"] = 1 + (df["factor"] -1) / 100

print(df)

# 월별 가중치를 위한 Simulation : 리챔 오믈레햄 340g

In [ ]:
# -*- coding: utf-8 -*-
import re, csv, os
from typing import List, Tuple, Dict

# -----------------------------------------
# 0) 후보 달 (2024-04 ~ 2025-06)
# -----------------------------------------
ALLOWED_MONTHS: List[str] = [
    "2024-04","2024-05","2024-06","2024-07","2024-08","2024-09",
    "2024-10","2024-11","2024-12","2025-01","2025-02","2025-03",
    "2025-04","2025-05","2025-06"
]

# -----------------------------------------
# 1) 속성 중요도
# -----------------------------------------
IMPORTANCE_ORDER = [
    "요리 여부·빈도",              # 속성8
    "월평균 가구 소득 수준",       # 속성5
    "가구원수",                   # 속성3
    "선호하는 통조림 햄 타입",     # 속성7
    "통조림 햄 주 사용 용도",      # 속성9
    "선호하는 통조림 햄 브랜드",   # 속성10
    "자녀 유무",                   # 속성4
    "구매 채널",                   # 속성6
    "나이",                        # 속성1
    "성별"                         # 속성2
]


# -----------------------------------------
# 1-1) 대상 제품/설명
# -----------------------------------------
PRODUCT_NAME = "리챔 오믈레햄 340g"
PRODUCT_DESC = (
    "‘리챔 오믈레햄 340g’은 오믈렛(Omelet)과 햄(Ham)의 합성 개념으로 개발된 신개념 통조림 햄 제품이다. "
    "저나트륨 설계를 통해 기존 햄 대비 부담을 줄였으며, 내열성 케첩 소스를 적용해 조리 시 풍미가 살아나는 것이 특징이다. "
    "한 끼 반찬이나 간편 도시락, 캠핑·야외활동 등 다양한 상황에서 활용하기 적합하며, "
    "부드러운 식감과 은은한 토마토 풍미가 더해져 아이들 간식부터 어른들의 반주 안주까지 폭넓게 어울린다. "
    "2025년 상반기에는 온라인몰 시식 리뷰 이벤트와 편의점 시식 프로모션을 통해 입소문을 확산시켰으며, "
    "소비자들에게 ‘간편하지만 특별한 한 끼’라는 이미지를 각인시키는 전략을 펼쳤다."
)

# -----------------------------------------
# 2) 프롬프트 (대상 제품 한정)
# -----------------------------------------
def build_single_month_prompt(persona_text: str, months: List[str]) -> str:
    months_line = ", ".join(months)
    importance_line = " > ".join(IMPORTANCE_ORDER)
    return f"""
너는 한국 소비자 신제품 햄 구매·섭취 행동을 분석하는 시장조사 전문가다.

[제품명]
{PRODUCT_NAME}

[제품 설명]
{PRODUCT_DESC}

[페르소나]
{persona_text.strip()}

[질문]
아래 달 중에서 이 페르소나가 위의 제품({PRODUCT_NAME})을 가장 많이 살 것 같은 달 1개만 고르라.
판단 시 아래 속성 중요도(상→하)를 우선 고려하라: {importance_line}

[선택 가능한 달(반드시 이 중에서만 선택)]
{months_line}

[출력 형식(다른 말 절대 금지)]
YYYY-MM
""".strip()

# -----------------------------------------
# 3) 응답 파서
# -----------------------------------------
RE_ISO_LINE  = re.compile(r'^\s*(20\d{2})-(0[1-9]|1[0-2])\s*$', re.MULTILINE)
RE_KOR_LINE  = re.compile(r'^\s*(20\d{2})\s*년\s*(1[0-2]|[1-9])\s*월\s*$', re.MULTILINE)

def parse_one_month(text: str, allowed: List[str]) -> str:
    for y, m in RE_ISO_LINE.findall(text):
        cand = f"{y}-{m}"
        if cand in allowed:
            return cand
    for y, m in RE_KOR_LINE.findall(text):
        cand = f"{y}-{int(m):02d}"
        if cand in allowed:
            return cand
    m = re.search(r'(20\d{2})-(0[1-9]|1[0-2])', text)
    if m:
        cand = f"{m.group(1)}-{m.group(2)}"
        if cand in allowed:
            return cand
    return ""

# -----------------------------------------
# 4) 페르소나 TXT 파싱
# -----------------------------------------
BLOCK_RE = re.compile(
    r"\*\*Persona_(\d{1,3})\*\*\s*(.+?)(?=(?:\n\s*\*\*Persona_\d{1,3}\*\*|\Z))",
    re.DOTALL
)

def parse_personas(text: str) -> List[Tuple[str, str]]:
    personas: List[Tuple[str, str]] = []
    for pid, body in BLOCK_RE.findall(text):
        personas.append((f"Persona_{int(pid):03d}", body.strip()))
    return personas

# -----------------------------------------
# 5) 단일 페르소나 질의
# -----------------------------------------
def ask_month_for_persona(
    pipe,
    persona_text: str,
    temperature: float = 1.20,
    top_p: float = 0.97,
    top_k: int = 0,
    max_new_tokens: int = 12,
    verbose: bool = False
) -> Tuple[str, str]:
    prompt = build_single_month_prompt(persona_text, ALLOWED_MONTHS)
    out = pipe(
        [{"role": "user", "content": prompt}],
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=temperature,
        top_p=top_p,
        top_k=top_k,
        repetition_penalty=1.0,
        return_full_text=False
    )
    raw = out[0].get("generated_text", out[0].get("text", "")).strip()
    month = parse_one_month(raw, ALLOWED_MONTHS) or ""
    if verbose:
        print(f"[RAW] {raw} -> [PARSED] {month or '(fail)'}")
    return month, raw

# -----------------------------------------
# 6) 전체 실행 루프
# -----------------------------------------
def run_survey_over_personas(
    pipe,
    personas_text: str,
    save_dir: str = ".",
    show_progress: bool = True,
    print_details: bool = True
) -> Dict[str, int]:
    os.makedirs(save_dir, exist_ok=True)

    personas = parse_personas(personas_text)
    if not personas:
        raise ValueError("페르소나를 찾지 못했으니 입력 형식을 확인하라.")

    month_counts: Dict[str, int] = {m: 0 for m in ALLOWED_MONTHS}
    total = len(personas)

    log_path = os.path.join(save_dir, "persona_choices.csv")
    with open(log_path, "w", newline="", encoding="utf-8") as f:
        wr = csv.writer(f)
        wr.writerow(["persona_id", "chosen_month", "raw_output"])

        for i, (pid, body) in enumerate(personas, start=1):
            if show_progress:
                print(f"[{i}/{total}] {pid} 질문 중...", flush=True)

            try:
                chosen, raw = ask_month_for_persona(pipe, body, verbose=False)
            except Exception as e:
                chosen, raw = "", f"__ERROR__: {e!r}"

            if chosen:
                month_counts[chosen] += 1
            wr.writerow([pid, chosen, raw])

            if show_progress and print_details:
                short_raw = (raw[:80] + "…") if len(raw) > 80 else raw
                print(f"   ↳ 선택: {chosen or '(파싱 실패)'} | 원문: {short_raw}", flush=True)

    cnt_path = os.path.join(save_dir, "month_counts_Omlet_340g.csv")
    with open(cnt_path, "w", newline="", encoding="utf-8") as f:
        wr = csv.writer(f)
        wr.writerow(["month", "count"])
        for m in ALLOWED_MONTHS:
            wr.writerow([m, month_counts[m]])

    print("\n[월별 선택 횟수]")
    for m in ALLOWED_MONTHS:
        print(f"{m}: {month_counts[m]}")

    print(f"\n저장 완료:\n- {log_path}\n- {cnt_path}")
    return month_counts

# -----------------------------------------
# 7) 실행
# -----------------------------------------
if __name__ == "__main__":
    month_counts = run_survey_over_personas(
        pipe,
        PERSONAS_RAW,
        save_dir=".",
        show_progress=True,
        print_details=True
    )


In [ ]:
import pandas as pd

# CSV 파일 불러오기
df = pd.read_csv("month_counts_Omlet_340g.csv", encoding="utf-8")

# 평균 count 계산
mean_count = df["count"].mean()

# factor 및 보정된 adj_factor 계산
df["factor"] = df["count"] / mean_count
df["adj_factor"] = 1 + (df["factor"] -1) / 100

print(df)